In [20]:
import sys
import os

sys.path.append(r"C:\Users\Manya Aggarwal\ApexCUDA")
os.chdir(r"C:\Users\Manya Aggarwal\ApexCUDA")

import pandas as pd
from backend.optimization_model import OptimizationModel

print(os.getcwd())

C:\Users\Manya Aggarwal\ApexCUDA


In [21]:
import os

print(os.getcwd())

C:\Users\Manya Aggarwal\ApexCUDA


In [22]:
flights = pd.read_csv("flights_1000.csv")
aircraft_120 = pd.read_csv("aircraft_120.csv")
crew_400 = pd.read_csv("crew_400.csv")
airport_capacity = pd.read_csv("airport_capacity.csv")

print("Flights:", flights.shape)
print("Aircraft:", aircraft_120.shape)
print("Crew:", crew_400.shape)
print("Airport:", airport_capacity.shape)

Flights: (789, 12)
Aircraft: (120, 6)
Crew: (400, 8)
Airport: (10, 5)


In [23]:
# PRESOLVER — CELL 1
# Build raw aircraft-flight candidate assignments

import pandas as pd

# Keep only available aircraft
available_aircraft = aircraft_120[
    aircraft_120["status"] == "AVAILABLE"
].copy()

candidate_rows = []

for _, flight in flights.iterrows():

    candidates = available_aircraft[
        (available_aircraft["aircraft_type"] == flight["aircraft_type"]) &
        (available_aircraft["capacity"] >= flight["passengers"]) &
        (available_aircraft["available_from"] <= flight["std"])
    ]

    for _, aircraft in candidates.iterrows():

        candidate_rows.append({
            "flight_id": flight["flight_id"],
            "aircraft_id": aircraft["aircraft_id"],
            "origin": flight["origin"],
            "destination": flight["destination"],
            "aircraft_type": flight["aircraft_type"],
            "std": flight["std"],
            "sta": flight["sta"],
            "passengers": flight["passengers"],
            "aircraft_capacity": aircraft["capacity"],
            "aircraft_location": aircraft["current_location"]
        })

aircraft_candidates = pd.DataFrame(candidate_rows)

print("Aircraft candidate table created.")
print()
print("Number of candidate assignments:",
      len(aircraft_candidates))

print("Number of flights:",
      aircraft_candidates["flight_id"].nunique())

print("Number of aircraft:",
      aircraft_candidates["aircraft_id"].nunique())

print("\nFirst 10 candidates:")
display(aircraft_candidates.head(10))

Aircraft candidate table created.

Number of candidate assignments: 31247
Number of flights: 789
Number of aircraft: 116

First 10 candidates:


,flight_id,aircraft_id,origin,destination,aircraft_type,std,sta,passengers,aircraft_capacity,aircraft_location
0,FL-1000,VT-AC101,DEL,GOI,A320,330,480,166,180,BOM
1,FL-1000,VT-AC102,DEL,GOI,A320,330,480,166,180,DEL
2,FL-1000,VT-AC103,DEL,GOI,A320,330,480,166,180,DEL
3,FL-1000,VT-AC104,DEL,GOI,A320,330,480,166,180,DEL
4,FL-1000,VT-AC106,DEL,GOI,A320,330,480,166,180,BOM
5,FL-1000,VT-AC107,DEL,GOI,A320,330,480,166,180,HYD
6,FL-1000,VT-AC108,DEL,GOI,A320,330,480,166,180,DEL
7,FL-1000,VT-AC109,DEL,GOI,A320,330,480,166,180,BOM
8,FL-1000,VT-AC110,DEL,GOI,A320,330,480,166,180,DEL
9,FL-1000,VT-AC111,DEL,GOI,A320,330,480,166,180,DEL


In [24]:
# PRESOLVER — CELL 2
# Filter aircraft candidates by initial aircraft location

initial_location_candidates = aircraft_candidates[
    aircraft_candidates["aircraft_location"] == aircraft_candidates["origin"]
].copy()

print("Before location filtering:", len(aircraft_candidates))
print("After location filtering:", len(initial_location_candidates))

removed = len(aircraft_candidates) - len(initial_location_candidates)

print("Candidates removed:", removed)

print(
    "Reduction:",
    round(100 * removed / len(aircraft_candidates), 2),
    "%"
)

print("\nFirst 10 remaining candidates:")
display(initial_location_candidates.head(10))

Before location filtering: 31247
After location filtering: 5501
Candidates removed: 25746
Reduction: 82.4 %

First 10 remaining candidates:


,flight_id,aircraft_id,origin,destination,aircraft_type,std,sta,passengers,aircraft_capacity,aircraft_location
1,FL-1000,VT-AC102,DEL,GOI,A320,330,480,166,180,DEL
2,FL-1000,VT-AC103,DEL,GOI,A320,330,480,166,180,DEL
3,FL-1000,VT-AC104,DEL,GOI,A320,330,480,166,180,DEL
6,FL-1000,VT-AC108,DEL,GOI,A320,330,480,166,180,DEL
8,FL-1000,VT-AC110,DEL,GOI,A320,330,480,166,180,DEL
9,FL-1000,VT-AC111,DEL,GOI,A320,330,480,166,180,DEL
11,FL-1000,VT-AC113,DEL,GOI,A320,330,480,166,180,DEL
12,FL-1000,VT-AC114,DEL,GOI,A320,330,480,166,180,DEL
15,FL-1000,VT-AC117,DEL,GOI,A320,330,480,166,180,DEL
18,FL-1000,VT-AC120,DEL,GOI,A320,330,480,166,180,DEL


In [25]:
# PRESOLVER — CELL 3
# Generate feasible flight-to-flight aircraft connections

TURNAROUND = 45  # minutes, based on the existing dataset

# Sort flights by departure time
flight_schedule = flights.sort_values("std").reset_index(drop=True)

aircraft_connection_rows = []

for i, flight_i in flight_schedule.iterrows():

    # Flights departing after flight_i arrives + turnaround
    possible_next = flight_schedule[
        flight_schedule["std"] >= flight_i["sta"] + TURNAROUND
    ]

    # Must depart from the previous flight's destination
    possible_next = possible_next[
        possible_next["origin"] == flight_i["destination"]
    ]

    for _, flight_j in possible_next.iterrows():

        # Aircraft types must be compatible
        if flight_i["aircraft_type"] != flight_j["aircraft_type"]:
            continue

        aircraft_connection_rows.append({
            "from_flight": flight_i["flight_id"],
            "to_flight": flight_j["flight_id"],
            "airport": flight_i["destination"],
            "aircraft_type": flight_i["aircraft_type"],
            "from_sta": flight_i["sta"],
            "to_std": flight_j["std"],
            "connection_gap": flight_j["std"] - flight_i["sta"]
        })

aircraft_connections = pd.DataFrame(aircraft_connection_rows)

print("Potential aircraft connections:", len(aircraft_connections))

print("\nConnection gap statistics:")
print(aircraft_connections["connection_gap"].describe())

print("\nConnections by aircraft type:")
print(
    aircraft_connections["aircraft_type"]
    .value_counts()
    .sort_index()
)

print("\nFirst 10 connections:")
display(aircraft_connections.head(10))

Potential aircraft connections: 10184

Connection gap statistics:
count    10184.000000
mean       338.691575
std        227.770409
min         45.000000
25%        143.000000
50%        300.000000
75%        506.000000
max        949.000000
Name: connection_gap, dtype: float64

Connections by aircraft type:
aircraft_type
A320     7099
A321     1962
ATR72     261
B737      862
Name: count, dtype: int64

First 10 connections:


,from_flight,to_flight,airport,aircraft_type,from_sta,to_std,connection_gap
0,FL-1000,FL-1149,GOI,A320,480,525,45
1,FL-1000,FL-1227,GOI,A320,480,612,132
2,FL-1000,FL-1246,GOI,A320,480,644,164
3,FL-1000,FL-1293,GOI,A320,480,706,226
4,FL-1000,FL-1300,GOI,A320,480,712,232
5,FL-1000,FL-1324,GOI,A320,480,747,267
6,FL-1000,FL-1325,GOI,A320,480,750,270
7,FL-1000,FL-1346,GOI,A320,480,776,296
8,FL-1000,FL-1358,GOI,A320,480,791,311
9,FL-1000,FL-1376,GOI,A320,480,820,340


In [26]:
# PRESOLVER — CELL 4
# Analyze flight connectivity in the aircraft network

incoming_counts = (
    aircraft_connections
    .groupby("to_flight")
    .size()
)

outgoing_counts = (
    aircraft_connections
    .groupby("from_flight")
    .size()
)

flights["possible_predecessors"] = (
    flights["flight_id"]
    .map(incoming_counts)
    .fillna(0)
    .astype(int)
)

flights["possible_successors"] = (
    flights["flight_id"]
    .map(outgoing_counts)
    .fillna(0)
    .astype(int)
)

print("Flights with no possible predecessor:",
      (flights["possible_predecessors"] == 0).sum())

print("Flights with no possible successor:",
      (flights["possible_successors"] == 0).sum())

print("\nPossible predecessors per flight:")
print(flights["possible_predecessors"].describe())

print("\nPossible successors per flight:")
print(flights["possible_successors"].describe())

print("\nFlights with the fewest predecessors:")
display(
    flights[
        [
            "flight_id",
            "origin",
            "destination",
            "aircraft_type",
            "std",
            "sta",
            "possible_predecessors",
            "possible_successors"
        ]
    ]
    .sort_values(["possible_predecessors", "possible_successors"])
    .head(15)
)

Flights with no possible predecessor: 116
Flights with no possible successor: 115

Possible predecessors per flight:
count    789.000000
mean      12.907478
std       14.113582
min        0.000000
25%        3.000000
50%        8.000000
75%       18.000000
max       69.000000
Name: possible_predecessors, dtype: float64

Possible successors per flight:
count    789.000000
mean      12.907478
std       14.117268
min        0.000000
25%        3.000000
50%        8.000000
75%       18.000000
max       69.000000
Name: possible_successors, dtype: float64

Flights with the fewest predecessors:


,flight_id,origin,destination,aircraft_type,std,sta,possible_predecessors,possible_successors
17,FL-1017,DEL,CCU,ATR72,347,477,0,3
64,FL-1064,BOM,MAA,ATR72,387,502,0,4
85,FL-1085,DEL,AMD,ATR72,401,491,0,5
15,FL-1015,BLR,HYD,B737,343,413,0,6
35,FL-1035,HYD,AMD,ATR72,358,472,0,6
106,FL-1106,DEL,MAA,B737,414,584,0,6
20,FL-1020,DEL,AMD,A321,348,438,0,7
54,FL-1054,BOM,HYD,ATR72,377,462,0,7
16,FL-1016,DEL,COK,A321,345,535,0,8
90,FL-1090,BOM,PNQ,A321,403,517,0,8


In [27]:
# PRESOLVER — CELL 5
# Check whether every flight can be entered by an aircraft

# Flights that can be the first flight of an aircraft
initial_flights = set(
    initial_location_candidates["flight_id"]
)

# Flights that can be reached from another flight
connection_flights = set(
    aircraft_connections["to_flight"]
)

# A flight is structurally reachable if:
# 1. an aircraft can start there, OR
# 2. another compatible flight can precede it

flights["aircraft_reachable"] = flights["flight_id"].apply(
    lambda f: (
        f in initial_flights
        or f in connection_flights
    )
)

print("Total flights:", len(flights))

print(
    "Flights reachable by aircraft network:",
    flights["aircraft_reachable"].sum()
)

print(
    "Flights with no aircraft entry:",
    (~flights["aircraft_reachable"]).sum()
)

print("\nUnreachable flights:")

display(
    flights[
        ~flights["aircraft_reachable"]
    ][[
        "flight_id",
        "origin",
        "destination",
        "aircraft_type",
        "std",
        "sta"
    ]]
)

Total flights: 789
Flights reachable by aircraft network: 789
Flights with no aircraft entry: 0

Unreachable flights:


,flight_id,origin,destination,aircraft_type,std,sta


In [28]:
# PRESOLVER — CELL 6
# Build raw crew-flight candidate assignments

available_crew = crew_400[
    crew_400["status"] == "AVAILABLE"
].copy()

crew_candidate_rows = []

for _, flight in flights.iterrows():

    # Find crew compatible with the aircraft type
    candidates = available_crew[
        (available_crew["type_rating"] == flight["aircraft_type"]) &
        (available_crew["available_from"] <= flight["std"])
    ]

    for _, crew in candidates.iterrows():

        crew_candidate_rows.append({
            "flight_id": flight["flight_id"],
            "crew_id": crew["crew_id"],
            "crew_type": crew["crew_type"],
            "type_rating": crew["type_rating"],
            "base_airport": crew["base_airport"],
            "flight_origin": flight["origin"],
            "flight_destination": flight["destination"],
            "std": flight["std"],
            "sta": flight["sta"],
            "max_duty_minutes": crew["max_duty_minutes"],
            "max_flights": crew["max_flights"]
        })

crew_candidates = pd.DataFrame(crew_candidate_rows)

print("Raw crew-flight candidate variables:",
      len(crew_candidates))

print("Unique flights:",
      crew_candidates["flight_id"].nunique())

print("Unique crew:",
      crew_candidates["crew_id"].nunique())

print("\nCandidates by crew type:")
print(
    crew_candidates["crew_type"]
    .value_counts()
)

print("\nCandidates by aircraft/crew rating:")
print(
    crew_candidates["type_rating"]
    .value_counts()
    .sort_index()
)

print("\nFirst 10 candidates:")
display(crew_candidates.head(10))

Raw crew-flight candidate variables: 106264
Unique flights: 789
Unique crew: 388

Candidates by crew type:
crew_type
CABIN_CREW       41943
CAPTAIN          32888
FIRST_OFFICER    31433
Name: count, dtype: int64

Candidates by aircraft/crew rating:
type_rating
A320     75077
A321     21560
ATR72     1932
B737      7695
Name: count, dtype: int64

First 10 candidates:


,flight_id,crew_id,crew_type,type_rating,base_airport,flight_origin,flight_destination,std,sta,max_duty_minutes,max_flights
0,FL-1000,CRW-0003,CAPTAIN,A320,HYD,DEL,GOI,330,480,480,4
1,FL-1000,CRW-0006,CAPTAIN,A320,BOM,DEL,GOI,330,480,480,4
2,FL-1000,CRW-0007,CAPTAIN,A320,DEL,DEL,GOI,330,480,480,4
3,FL-1000,CRW-0008,CAPTAIN,A320,DEL,DEL,GOI,330,480,480,4
4,FL-1000,CRW-0013,CAPTAIN,A320,DEL,DEL,GOI,330,480,480,4
5,FL-1000,CRW-0014,CAPTAIN,A320,BOM,DEL,GOI,330,480,480,4
6,FL-1000,CRW-0016,CAPTAIN,A320,HYD,DEL,GOI,330,480,480,4
7,FL-1000,CRW-0017,CAPTAIN,A320,BOM,DEL,GOI,330,480,480,4
8,FL-1000,CRW-0018,CAPTAIN,A320,HYD,DEL,GOI,330,480,480,4
9,FL-1000,CRW-0019,CAPTAIN,A320,DEL,DEL,GOI,330,480,480,4


In [29]:
# Load the flights dataset

import pandas as pd

flights = pd.read_csv("flights_1000.csv")

print("Flights shape:", flights.shape)
display(flights.head())

Flights shape: (789, 12)


,flight_id,origin,destination,aircraft_type,assigned_aircraft,std,sta,flight_duration,passengers,forced_delay,c_cancel,c_delay_per_min
0,FL-1000,DEL,GOI,A320,VT-AC135,330,480,150,166,0,690440,665
1,FL-1001,DEL,BLR,A320,VT-AC126,331,491,160,175,0,720475,785
2,FL-1002,DEL,BOM,ATR72,VT-AC220,331,456,125,65,0,378492,930
3,FL-1003,DEL,CCU,A320,VT-AC114,333,463,130,148,0,699701,719
4,FL-1004,HYD,DEL,ATR72,VT-AC215,333,463,130,61,15,381809,732


In [30]:
# PRESOLVER — CELL 7
# Find overlapping flight pairs for crew scheduling

flight_times = flights[
    ["flight_id", "origin", "destination", "aircraft_type", "std", "sta"]
].sort_values("std").reset_index(drop=True)

crew_conflict_rows = []

for i in range(len(flight_times)):

    flight_i = flight_times.iloc[i]

    # Only examine flights that start before flight_i ends
    for j in range(i + 1, len(flight_times)):

        flight_j = flight_times.iloc[j]

        # Since flights are sorted by std, once this is true,
        # later flights will also start after flight_i ends.
        if flight_j["std"] >= flight_i["sta"]:
            break

        # Flights overlap in time
        crew_conflict_rows.append({
            "flight_1": flight_i["flight_id"],
            "flight_2": flight_j["flight_id"],
            "flight_1_type": flight_i["aircraft_type"],
            "flight_2_type": flight_j["aircraft_type"],
            "flight_1_origin": flight_i["origin"],
            "flight_2_origin": flight_j["origin"],
            "overlap_start": flight_j["std"],
            "overlap_end": min(
                flight_i["sta"],
                flight_j["sta"]
            )
        })

crew_conflicts = pd.DataFrame(crew_conflict_rows)

print("Overlapping flight pairs:", len(crew_conflicts))

print("\nFirst 10 conflicts:")
display(crew_conflicts.head(10))

print("\nFlights involved in at least one conflict:")

conflict_flights = set(
    crew_conflicts["flight_1"]
).union(
    set(crew_conflicts["flight_2"])
)

print(len(conflict_flights), "of", len(flights))

Overlapping flight pairs: 61120

First 10 conflicts:


,flight_1,flight_2,flight_1_type,flight_2_type,flight_1_origin,flight_2_origin,overlap_start,overlap_end
0,FL-1000,FL-1001,A320,A320,DEL,DEL,331,480
1,FL-1000,FL-1002,A320,ATR72,DEL,DEL,331,456
2,FL-1000,FL-1003,A320,A320,DEL,DEL,333,463
3,FL-1000,FL-1004,A320,ATR72,DEL,HYD,333,463
4,FL-1000,FL-1005,A320,A320,DEL,BOM,335,430
5,FL-1000,FL-1006,A320,A321,DEL,BOM,336,431
6,FL-1000,FL-1007,A320,A320,DEL,DEL,336,461
7,FL-1000,FL-1008,A320,A321,DEL,DEL,336,480
8,FL-1000,FL-1009,A320,A320,DEL,BLR,337,437
9,FL-1000,FL-1010,A320,A320,DEL,DEL,340,480



Flights involved in at least one conflict:
789 of 789


In [31]:
# PRESOLVER — CELL 8 (CORRECTED)
# Count feasible crew candidates per flight and crew role

crew_counts = (
    crew_candidates
    .groupby(["flight_id", "crew_type"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

print("Crew types found:",
      crew_candidates["crew_type"].unique())

print("\nFlights:", len(crew_counts))

display(crew_counts.head(10))

print("\nMinimum candidates per role:")

for role in crew_candidates["crew_type"].unique():
    print(role + ":", crew_counts[role].min())

print("\nAverage candidates per role:")

for role in crew_candidates["crew_type"].unique():
    print(role + ":", round(crew_counts[role].mean(), 2))

print("\nFlights with zero candidates:")

for role in crew_candidates["crew_type"].unique():
    print(role + ":", (crew_counts[role] == 0).sum())

Crew types found: <StringArray>
['CAPTAIN', 'FIRST_OFFICER', 'CABIN_CREW']
Length: 3, dtype: str

Flights: 789


crew_type,flight_id,CABIN_CREW,CAPTAIN,FIRST_OFFICER
0,FL-1000,78,59,56
1,FL-1001,78,59,56
2,FL-1002,15,3,10
3,FL-1003,78,59,56
4,FL-1004,15,3,10
5,FL-1005,78,59,56
6,FL-1006,36,40,34
7,FL-1007,78,59,56
8,FL-1008,36,40,34
9,FL-1009,78,59,56



Minimum candidates per role:
CAPTAIN: 3
FIRST_OFFICER: 10
CABIN_CREW: 15

Average candidates per role:
CAPTAIN: 41.68
FIRST_OFFICER: 39.84
CABIN_CREW: 53.16

Flights with zero candidates:
CAPTAIN: 0
FIRST_OFFICER: 0
CABIN_CREW: 0


In [32]:
# PRESOLVER — CELL 9
# Generate temporally feasible crew flight-to-flight connections

crew_flights = flights[
    ["flight_id", "origin", "destination", "aircraft_type", "std", "sta"]
].sort_values("std").reset_index(drop=True)

crew_connections = []

MIN_TURNAROUND = 45

for i in range(len(crew_flights)):

    flight_i = crew_flights.iloc[i]

    for j in range(i + 1, len(crew_flights)):

        flight_j = crew_flights.iloc[j]

        # Since flights are sorted by std, once this condition
        # fails for the first time, later flights will also fail.
        if flight_j["std"] < flight_i["sta"] + MIN_TURNAROUND:
            continue

        # Crew must be able to physically move from the
        # previous flight's destination to the next flight's origin.
        if flight_i["destination"] != flight_j["origin"]:
            continue

        crew_connections.append({
            "flight_1": flight_i["flight_id"],
            "flight_2": flight_j["flight_id"],
            "flight_1_type": flight_i["aircraft_type"],
            "flight_2_type": flight_j["aircraft_type"],
            "destination_1": flight_i["destination"],
            "origin_2": flight_j["origin"],
            "gap": flight_j["std"] - flight_i["sta"]
        })

crew_connections = pd.DataFrame(crew_connections)

print("Feasible crew flight-to-flight connections:",
      len(crew_connections))

print("\nFirst 10 connections:")
display(crew_connections.head(10))

if len(crew_connections) > 0:
    print("\nGap statistics:")
    print(crew_connections["gap"].describe())

Feasible crew flight-to-flight connections: 28679

First 10 connections:


,flight_1,flight_2,flight_1_type,flight_2_type,destination_1,origin_2,gap
0,FL-1000,FL-1149,A320,A320,GOI,GOI,45
1,FL-1000,FL-1165,A320,A321,GOI,GOI,58
2,FL-1000,FL-1210,A320,B737,GOI,GOI,107
3,FL-1000,FL-1217,A320,B737,GOI,GOI,114
4,FL-1000,FL-1227,A320,A320,GOI,GOI,132
5,FL-1000,FL-1242,A320,A321,GOI,GOI,154
6,FL-1000,FL-1246,A320,A320,GOI,GOI,164
7,FL-1000,FL-1269,A320,B737,GOI,GOI,200
8,FL-1000,FL-1293,A320,A320,GOI,GOI,226
9,FL-1000,FL-1298,A320,B737,GOI,GOI,231



Gap statistics:
count    28679.000000
mean       350.920430
std        224.561949
min         45.000000
25%        160.000000
50%        312.000000
75%        516.000000
max        953.000000
Name: gap, dtype: float64


In [33]:
# PRESOLVER — CELL 10
# Check crew-network reachability for every flight

# Flights that can have another flight immediately before them
crew_predecessor_counts = (
    crew_connections
    .groupby("flight_2")
    .size()
    .rename("possible_crew_predecessors")
)

# Flights that can have another flight immediately after them
crew_successor_counts = (
    crew_connections
    .groupby("flight_1")
    .size()
    .rename("possible_crew_successors")
)

crew_reachability = flights[
    ["flight_id", "origin", "destination", "aircraft_type", "std", "sta"]
].copy()

crew_reachability = crew_reachability.merge(
    crew_predecessor_counts,
    left_on="flight_id",
    right_index=True,
    how="left"
)

crew_reachability = crew_reachability.merge(
    crew_successor_counts,
    left_on="flight_id",
    right_index=True,
    how="left"
)

crew_reachability[
    ["possible_crew_predecessors", "possible_crew_successors"]
] = crew_reachability[
    ["possible_crew_predecessors", "possible_crew_successors"]
].fillna(0).astype(int)

print("Flights with no possible predecessor:",
      (crew_reachability["possible_crew_predecessors"] == 0).sum())

print("Flights with no possible successor:",
      (crew_reachability["possible_crew_successors"] == 0).sum())

print("\nPredecessor statistics:")
print(crew_reachability["possible_crew_predecessors"].describe())

print("\nSuccessor statistics:")
print(crew_reachability["possible_crew_successors"].describe())

print("\nFlights with no predecessor:")
display(
    crew_reachability[
        crew_reachability["possible_crew_predecessors"] == 0
    ].head(10)
)

print("\nFlights with no successor:")
display(
    crew_reachability[
        crew_reachability["possible_crew_successors"] == 0
    ].head(10)
)

Flights with no possible predecessor: 116
Flights with no possible successor: 115

Predecessor statistics:
count    789.000000
mean      36.348542
std       33.756352
min        0.000000
25%        9.000000
50%       28.000000
75%       52.000000
max      135.000000
Name: possible_crew_predecessors, dtype: float64

Successor statistics:
count    789.000000
mean      36.348542
std       33.748945
min        0.000000
25%        9.000000
50%       28.000000
75%       52.000000
max      135.000000
Name: possible_crew_successors, dtype: float64

Flights with no predecessor:


,flight_id,origin,destination,aircraft_type,std,sta,possible_crew_predecessors,possible_crew_successors
0,FL-1000,DEL,GOI,A320,330,480,0,50
1,FL-1001,DEL,BLR,A320,331,491,0,126
2,FL-1002,DEL,BOM,ATR72,331,456,0,116
3,FL-1003,DEL,CCU,A320,333,463,0,48
4,FL-1004,HYD,DEL,ATR72,333,463,0,97
5,FL-1005,BOM,BLR,A320,335,430,0,135
6,FL-1006,BOM,BLR,A321,336,431,0,134
7,FL-1007,DEL,BOM,A320,336,461,0,115
8,FL-1008,DEL,BLR,A321,336,496,0,124
9,FL-1009,BLR,BOM,A320,337,437,0,118



Flights with no successor:


,flight_id,origin,destination,aircraft_type,std,sta,possible_crew_predecessors,possible_crew_successors
642,FL-1642,BLR,DEL,A320,1184,1349,100,0
643,FL-1643,BOM,CCU,A320,1185,1340,93,0
652,FL-1652,CCU,DEL,A321,1196,1336,37,0
660,FL-1660,BOM,DEL,A320,1205,1335,99,0
661,FL-1661,PNQ,BOM,A320,1208,1337,32,0
663,FL-1663,DEL,COK,A321,1210,1400,77,0
665,FL-1665,DEL,GOI,B737,1213,1363,78,0
670,FL-1670,MAA,BOM,ATR72,1219,1339,37,0
672,FL-1672,CCU,AMD,A321,1222,1346,38,0
674,FL-1674,HYD,GOI,A320,1226,1361,39,0


In [34]:
# PRESOLVER — CELL 11
# Prepare crew assignment candidate sets

crew_candidate_sets = {}

for role in ["CAPTAIN", "FIRST_OFFICER", "CABIN_CREW"]:

    role_candidates = crew_candidates[
        crew_candidates["crew_type"] == role
    ].copy()

    role_candidates = role_candidates[
        ["flight_id", "crew_id", "crew_type", "type_rating"]
    ].drop_duplicates()

    crew_candidate_sets[role] = role_candidates

    print(f"\n{role}")
    print("Candidate assignments:", len(role_candidates))
    print("Unique flights:", role_candidates["flight_id"].nunique())
    print("Unique crew:", role_candidates["crew_id"].nunique())

print(
    "\nTotal crew assignment variables:",
    sum(len(df) for df in crew_candidate_sets.values())
)


CAPTAIN
Candidate assignments: 32888
Unique flights: 789
Unique crew: 116

FIRST_OFFICER
Candidate assignments: 31433
Unique flights: 789
Unique crew: 117

CABIN_CREW
Candidate assignments: 41943
Unique flights: 789
Unique crew: 155

Total crew assignment variables: 106264


In [35]:
# PRESOLVER — CELL 12
# Determine aircraft-reachable flights

# Flights that can be the first flight of an aircraft
initial_aircraft_flights = set(
    initial_location_candidates["flight_id"]
)

# Flights that can be reached after another aircraft flight
connection_reachable_flights = set(
    aircraft_connections["to_flight"]
)

# A flight is reachable if:
# 1. An aircraft can start there from its initial location, OR
# 2. Another feasible aircraft flight can precede it.

aircraft_reachable_ids = (
    initial_aircraft_flights
    .union(connection_reachable_flights)
)

flights["aircraft_reachable"] = (
    flights["flight_id"].isin(aircraft_reachable_ids)
)

print("Total flights:", len(flights))

print(
    "Aircraft-reachable flights:",
    flights["aircraft_reachable"].sum()
)

print(
    "Aircraft-unreachable flights:",
    (~flights["aircraft_reachable"]).sum()
)

print("\nUnreachable flights:")

display(
    flights[
        ~flights["aircraft_reachable"]
    ][
        [
            "flight_id",
            "origin",
            "destination",
            "aircraft_type",
            "std",
            "sta"
        ]
    ]
)

Total flights: 789
Aircraft-reachable flights: 789
Aircraft-unreachable flights: 0

Unreachable flights:


,flight_id,origin,destination,aircraft_type,std,sta


In [36]:
# PRESOLVER — CELL 13
# Build aircraft assignment candidates with network information

aircraft_reduced_candidates = aircraft_candidates[
    [
        "flight_id",
        "aircraft_id",
        "aircraft_type",
        "origin",
        "destination",
        "std",
        "sta",
        "passengers"
    ]
].drop_duplicates().copy()

# Mark whether this aircraft can start its schedule with this flight
initial_pairs = set(
    zip(
        initial_location_candidates["flight_id"],
        initial_location_candidates["aircraft_id"]
    )
)

aircraft_reduced_candidates["can_start"] = [
    (f, a) in initial_pairs
    for f, a in zip(
        aircraft_reduced_candidates["flight_id"],
        aircraft_reduced_candidates["aircraft_id"]
    )
]

print("Raw aircraft candidates:",
      len(aircraft_reduced_candidates))

print(
    "Candidates that can be initial assignments:",
    aircraft_reduced_candidates["can_start"].sum()
)

print(
    "Candidates requiring a predecessor:",
    (~aircraft_reduced_candidates["can_start"]).sum()
)

display(aircraft_reduced_candidates.head(10))

Raw aircraft candidates: 31247
Candidates that can be initial assignments: 5501
Candidates requiring a predecessor: 25746


,flight_id,aircraft_id,aircraft_type,origin,destination,std,sta,passengers,can_start
0,FL-1000,VT-AC101,A320,DEL,GOI,330,480,166,False
1,FL-1000,VT-AC102,A320,DEL,GOI,330,480,166,True
2,FL-1000,VT-AC103,A320,DEL,GOI,330,480,166,True
3,FL-1000,VT-AC104,A320,DEL,GOI,330,480,166,True
4,FL-1000,VT-AC106,A320,DEL,GOI,330,480,166,False
5,FL-1000,VT-AC107,A320,DEL,GOI,330,480,166,False
6,FL-1000,VT-AC108,A320,DEL,GOI,330,480,166,True
7,FL-1000,VT-AC109,A320,DEL,GOI,330,480,166,False
8,FL-1000,VT-AC110,A320,DEL,GOI,330,480,166,True
9,FL-1000,VT-AC111,A320,DEL,GOI,330,480,166,True


In [37]:
# PRESOLVER — CELL 14
# Check whether each aircraft-flight candidate has a feasible predecessor

# Create lookup of (to_flight, aircraft_type) combinations
# that have at least one possible predecessor.
aircraft_predecessor_flights = set(
    zip(
        aircraft_connections["to_flight"],
        aircraft_connections["aircraft_type"]
    )
)

aircraft_reduced_candidates["has_network_predecessor"] = [
    (f, t) in aircraft_predecessor_flights
    for f, t in zip(
        aircraft_reduced_candidates["flight_id"],
        aircraft_reduced_candidates["aircraft_type"]
    )
]

# A candidate is structurally reachable if:
# - the aircraft can start with this flight, OR
# - the flight can be reached through the aircraft network
aircraft_reduced_candidates["structurally_reachable"] = (
    aircraft_reduced_candidates["can_start"]
    |
    aircraft_reduced_candidates["has_network_predecessor"]
)

print(
    "Total aircraft candidates:",
    len(aircraft_reduced_candidates)
)

print(
    "Structurally reachable candidates:",
    aircraft_reduced_candidates["structurally_reachable"].sum()
)

print(
    "Structurally unreachable candidates:",
    (~aircraft_reduced_candidates["structurally_reachable"]).sum()
)

print("\nCandidates by status:")

print(
    "Can start:",
    aircraft_reduced_candidates["can_start"].sum()
)

print(
    "Needs predecessor:",
    (~aircraft_reduced_candidates["can_start"]).sum()
)

print(
    "Has network predecessor:",
    aircraft_reduced_candidates["has_network_predecessor"].sum()
)

Total aircraft candidates: 31247
Structurally reachable candidates: 28217
Structurally unreachable candidates: 3030

Candidates by status:
Can start: 5501
Needs predecessor: 25746
Has network predecessor: 26657


In [38]:
# PRESOLVER — CELL 15
# Aircraft-specific network reachability

# Sort flights chronologically
sorted_flights = flights[
    [
        "flight_id",
        "origin",
        "destination",
        "aircraft_type",
        "std",
        "sta"
    ]
].sort_values("std").reset_index(drop=True)

# Build outgoing connection lookup:
# from_flight -> list of possible next flights
next_flights = {}

for _, row in aircraft_connections.iterrows():
    next_flights.setdefault(
        row["from_flight"], []
    ).append(row["to_flight"])

# Aircraft-specific reachable flights
reachable_aircraft_flights = set()

for _, aircraft in aircraft_120.iterrows():

    aircraft_id = aircraft["aircraft_id"]
    aircraft_type = aircraft["aircraft_type"]
    current_location = aircraft["current_location"]

    # Flights that this aircraft can perform as its first flight
    start_flights = sorted_flights[
        (sorted_flights["aircraft_type"] == aircraft_type) &
        (sorted_flights["origin"] == current_location)
    ]

    visited = set()
    stack = list(start_flights["flight_id"])

    # Traverse the flight connection network
    while stack:

        flight_id = stack.pop()

        if flight_id in visited:
            continue

        visited.add(flight_id)

        # Record this specific aircraft-flight pair
        reachable_aircraft_flights.add(
            (aircraft_id, flight_id)
        )

        # Follow valid next-flight connections
        for next_flight in next_flights.get(flight_id, []):
            if next_flight not in visited:
                stack.append(next_flight)

# Check every raw candidate
aircraft_reduced_candidates["aircraft_network_reachable"] = [
    (a, f) in reachable_aircraft_flights
    for f, a in zip(
        aircraft_reduced_candidates["flight_id"],
        aircraft_reduced_candidates["aircraft_id"]
    )
]

print(
    "Raw aircraft candidates:",
    len(aircraft_reduced_candidates)
)

print(
    "Aircraft-specific reachable:",
    aircraft_reduced_candidates[
        "aircraft_network_reachable"
    ].sum()
)

print(
    "Aircraft-specific unreachable:",
    (~aircraft_reduced_candidates[
        "aircraft_network_reachable"
    ]).sum()
)

print("\nReduction:",
      round(
          100 * (
              (~aircraft_reduced_candidates[
                  "aircraft_network_reachable"
              ]).sum()
              / len(aircraft_reduced_candidates)
          ),
          2
      ),
      "%"
    )

Raw aircraft candidates: 31247
Aircraft-specific reachable: 26488
Aircraft-specific unreachable: 4759

Reduction: 15.23 %


In [39]:
# PRESOLVER — CELL 16
# Create the final reduced aircraft candidate set

aircraft_reduced_candidates = (
    aircraft_reduced_candidates[
        aircraft_reduced_candidates["aircraft_network_reachable"]
    ]
    .copy()
    .reset_index(drop=True)
)

print("Original aircraft candidates:", 31247)

print(
    "Reduced aircraft candidates:",
    len(aircraft_reduced_candidates)
)

print(
    "Variables removed:",
    31247 - len(aircraft_reduced_candidates)
)

print(
    "Reduction:",
    round(
        100 * (1 - len(aircraft_reduced_candidates) / 31247),
        2
    ),
    "%"
)

print("\nUnique flights:",
      aircraft_reduced_candidates["flight_id"].nunique())

print("Unique aircraft:",
      aircraft_reduced_candidates["aircraft_id"].nunique())

print("\nFirst 10 reduced candidates:")
display(aircraft_reduced_candidates.head(10))

Original aircraft candidates: 31247
Reduced aircraft candidates: 26488
Variables removed: 4759
Reduction: 15.23 %

Unique flights: 789
Unique aircraft: 116

First 10 reduced candidates:


,flight_id,aircraft_id,aircraft_type,origin,destination,std,sta,passengers,can_start,has_network_predecessor,structurally_reachable,aircraft_network_reachable
0,FL-1000,VT-AC102,A320,DEL,GOI,330,480,166,True,False,True,True
1,FL-1000,VT-AC103,A320,DEL,GOI,330,480,166,True,False,True,True
2,FL-1000,VT-AC104,A320,DEL,GOI,330,480,166,True,False,True,True
3,FL-1000,VT-AC108,A320,DEL,GOI,330,480,166,True,False,True,True
4,FL-1000,VT-AC110,A320,DEL,GOI,330,480,166,True,False,True,True
5,FL-1000,VT-AC111,A320,DEL,GOI,330,480,166,True,False,True,True
6,FL-1000,VT-AC113,A320,DEL,GOI,330,480,166,True,False,True,True
7,FL-1000,VT-AC114,A320,DEL,GOI,330,480,166,True,False,True,True
8,FL-1000,VT-AC117,A320,DEL,GOI,330,480,166,True,False,True,True
9,FL-1000,VT-AC120,A320,DEL,GOI,330,480,166,True,False,True,True


In [40]:
# PRESOLVER — CELL 17
# Clean and validate reduced aircraft candidate pairs

# Keep only one row per (flight, aircraft) decision variable
aircraft_reduced_candidates = (
    aircraft_reduced_candidates[
        [
            "flight_id",
            "aircraft_id",
            "aircraft_type",
            "origin",
            "destination",
            "std",
            "sta",
            "passengers",
            "can_start",
            "has_network_predecessor"
        ]
    ]
    .drop_duplicates(
        subset=["flight_id", "aircraft_id"]
    )
    .reset_index(drop=True)
)

# Count candidates available for each flight
aircraft_candidates_per_flight = (
    aircraft_reduced_candidates
    .groupby("flight_id")
    .size()
)

print("Total reduced aircraft variables:",
      len(aircraft_reduced_candidates))

print("Duplicate (flight, aircraft) pairs:",
      len(aircraft_reduced_candidates)
      - aircraft_reduced_candidates[
          ["flight_id", "aircraft_id"]
      ].drop_duplicates().shape[0])

print("\nCandidates per flight:")
print(
    aircraft_candidates_per_flight.describe()
)

print("\nFlights with minimum candidates:",
      aircraft_candidates_per_flight.min())

print("Flights with maximum candidates:",
      aircraft_candidates_per_flight.max())

print("Flights with zero candidates:",
      len(
          set(flights["flight_id"])
          - set(aircraft_candidates_per_flight.index)
      )
)

print("\nAircraft candidates by type:")
print(
    aircraft_reduced_candidates[
        ["aircraft_type", "aircraft_id"]
    ]
    .drop_duplicates()
    ["aircraft_type"]
    .value_counts()
    .sort_index()
)

Total reduced aircraft variables: 26488
Duplicate (flight, aircraft) pairs: 0

Candidates per flight:
count    789.000000
mean      33.571610
std       19.492326
min        1.000000
25%       18.000000
50%       29.000000
75%       57.000000
max       57.000000
dtype: float64

Flights with minimum candidates: 1
Flights with maximum candidates: 57
Flights with zero candidates: 0

Aircraft candidates by type:
aircraft_type
A320     57
A321     29
ATR72    10
B737     20
Name: count, dtype: int64


In [41]:
# PRESOLVER — CELL 18
# Identify flights with exactly one feasible aircraft

candidate_counts = (
    aircraft_reduced_candidates
    .groupby("flight_id")
    .size()
    .reset_index(name="num_aircraft_candidates")
)

forced_aircraft_flights = candidate_counts[
    candidate_counts["num_aircraft_candidates"] == 1
].copy()

# Get the actual aircraft for those flights
forced_aircraft_flights = forced_aircraft_flights.merge(
    aircraft_reduced_candidates[
        ["flight_id", "aircraft_id", "aircraft_type",
         "origin", "destination", "std", "sta"]
    ],
    on="flight_id",
    how="left"
)

print("Flights with exactly one aircraft candidate:",
      len(forced_aircraft_flights))

display(forced_aircraft_flights)

Flights with exactly one aircraft candidate: 1


,flight_id,num_aircraft_candidates,aircraft_id,aircraft_type,origin,destination,std,sta
0,FL-1029,1,VT-AC200,B737,HYD,BOM,354,439


In [42]:
# PRESOLVER — CELL 19
# Identify aircraft that have only one feasible flight candidate

aircraft_candidate_counts = (
    aircraft_reduced_candidates
    .groupby("aircraft_id")
    .size()
    .reset_index(name="num_flight_candidates")
)

single_flight_aircraft = aircraft_candidate_counts[
    aircraft_candidate_counts["num_flight_candidates"] == 1
].copy()

single_flight_aircraft = single_flight_aircraft.merge(
    aircraft_reduced_candidates[
        [
            "aircraft_id",
            "flight_id",
            "aircraft_type",
            "origin",
            "destination",
            "std",
            "sta"
        ]
    ],
    on="aircraft_id",
    how="left"
)

print(
    "Aircraft with exactly one feasible flight:",
    len(single_flight_aircraft)
)

display(single_flight_aircraft)

Aircraft with exactly one feasible flight: 0


,aircraft_id,num_flight_candidates,flight_id,aircraft_type,origin,destination,std,sta


In [43]:
# PRESOLVER — CELL 20
# Build aircraft-specific feasible sequencing arcs

# Candidate assignments: (flight, aircraft)
candidate_pairs = aircraft_reduced_candidates[
    ["flight_id", "aircraft_id", "aircraft_type"]
].drop_duplicates()

# Connections between flights
connection_pairs = aircraft_connections[
    [
        "from_flight",
        "to_flight",
        "aircraft_type",
        "connection_gap"
    ]
].drop_duplicates()

# Match each flight connection with aircraft that can perform
# both flights
aircraft_sequence_arcs = connection_pairs.merge(
    candidate_pairs,
    left_on=["from_flight", "aircraft_type"],
    right_on=["flight_id", "aircraft_type"],
    how="inner"
)

aircraft_sequence_arcs = aircraft_sequence_arcs.merge(
    candidate_pairs,
    left_on=["to_flight", "aircraft_type", "aircraft_id"],
    right_on=["flight_id", "aircraft_type", "aircraft_id"],
    how="inner",
    suffixes=("_from", "_to")
)

aircraft_sequence_arcs = aircraft_sequence_arcs[
    [
        "from_flight",
        "to_flight",
        "aircraft_id",
        "aircraft_type",
        "connection_gap"
    ]
].drop_duplicates().reset_index(drop=True)

print("Aircraft-specific sequencing arcs:",
      len(aircraft_sequence_arcs))

print("\nUnique aircraft:",
      aircraft_sequence_arcs["aircraft_id"].nunique())

print("Unique from-flights:",
      aircraft_sequence_arcs["from_flight"].nunique())

print("Unique to-flights:",
      aircraft_sequence_arcs["to_flight"].nunique())

print("\nConnection gap statistics:")
print(
    aircraft_sequence_arcs["connection_gap"].describe()
)

print("\nFirst 10 sequencing arcs:")
display(aircraft_sequence_arcs.head(10))

Aircraft-specific sequencing arcs: 338468

Unique aircraft: 116
Unique from-flights: 674
Unique to-flights: 673

Connection gap statistics:
count    338468.000000
mean        297.128254
std         201.821886
min          45.000000
25%         128.000000
50%         255.000000
75%         434.000000
max         949.000000
Name: connection_gap, dtype: float64

First 10 sequencing arcs:


,from_flight,to_flight,aircraft_id,aircraft_type,connection_gap
0,FL-1000,FL-1149,VT-AC102,A320,45
1,FL-1000,FL-1149,VT-AC103,A320,45
2,FL-1000,FL-1149,VT-AC104,A320,45
3,FL-1000,FL-1149,VT-AC108,A320,45
4,FL-1000,FL-1149,VT-AC110,A320,45
5,FL-1000,FL-1149,VT-AC111,A320,45
6,FL-1000,FL-1149,VT-AC113,A320,45
7,FL-1000,FL-1149,VT-AC114,A320,45
8,FL-1000,FL-1149,VT-AC117,A320,45
9,FL-1000,FL-1149,VT-AC120,A320,45


In [44]:
# PRESOLVER — CELL 21
# Analyze aircraft-specific sequencing arc density

arcs_per_aircraft = (
    aircraft_sequence_arcs
    .groupby("aircraft_id")
    .size()
)

outgoing_arcs_per_flight = (
    aircraft_sequence_arcs
    .groupby(["aircraft_id", "from_flight"])
    .size()
)

incoming_arcs_per_flight = (
    aircraft_sequence_arcs
    .groupby(["aircraft_id", "to_flight"])
    .size()
)

print("Total sequencing arcs:",
      len(aircraft_sequence_arcs))

print("\nArcs per aircraft:")
print(arcs_per_aircraft.describe())

print("\nOutgoing choices per aircraft-flight:")
print(outgoing_arcs_per_flight.describe())

print("\nIncoming choices per aircraft-flight:")
print(incoming_arcs_per_flight.describe())

print("\nAircraft with most sequencing arcs:")
print(
    arcs_per_aircraft
    .sort_values(ascending=False)
    .head(10)
)

print("\nAircraft-specific arcs with only one outgoing choice:",
      (outgoing_arcs_per_flight == 1).sum())

print("Aircraft-specific arcs with only one incoming choice:",
      (incoming_arcs_per_flight == 1).sum())

Total sequencing arcs: 338468

Arcs per aircraft:
count     116.000000
mean     2917.827586
std      2190.281547
min       164.000000
25%       649.000000
50%      1475.000000
75%      5282.000000
max      5411.000000
dtype: float64

Outgoing choices per aircraft-flight:
count    21918.000000
mean        15.442467
std         13.324134
min          1.000000
25%          5.000000
50%         12.000000
75%         21.000000
max         69.000000
dtype: float64

Incoming choices per aircraft-flight:
count    24318.000000
mean        13.918414
std         12.682848
min          1.000000
25%          4.000000
50%         10.000000
75%         19.000000
max         59.000000
dtype: float64

Aircraft with most sequencing arcs:
aircraft_id
VT-AC102    5411
VT-AC103    5411
VT-AC104    5411
VT-AC108    5411
VT-AC117    5411
VT-AC113    5411
VT-AC111    5411
VT-AC110    5411
VT-AC114    5411
VT-AC120    5411
dtype: int64

Aircraft-specific arcs with only one outgoing choice: 1151
Aircraft-specif

In [45]:
# PRESOLVER — CELL 22
# Create flight-level decision variable definitions

flight_variables = flights[
    [
        "flight_id",
        "std",
        "sta",
        "forced_delay",
        "c_cancel",
        "c_delay_per_min",
        "passengers",
        "aircraft_type",
        "origin",
        "destination"
    ]
].copy()

# Cancellation variable:
# y_f = 1 -> cancelled
# y_f = 0 -> operated
flight_variables["cancel_lb"] = 0
flight_variables["cancel_ub"] = 1
flight_variables["cancel_integrality"] = 1

# Delay variable:
# d_f >= forced_delay for operated flights.
# For now, define a practical upper bound of 180 minutes.
flight_variables["delay_lb"] = flight_variables["forced_delay"]
flight_variables["delay_ub"] = 180
flight_variables["delay_integrality"] = 0

print("Flights:", len(flight_variables))

print("\nCancellation variables:",
      len(flight_variables))

print("Delay variables:",
      len(flight_variables))

print("\nForced-delay flights:",
      (flight_variables["forced_delay"] > 0).sum())

print("Maximum forced delay:",
      flight_variables["forced_delay"].max())

print("\nFirst 10 flight variables:")
display(flight_variables.head(10))

Flights: 789

Cancellation variables: 789
Delay variables: 789

Forced-delay flights: 60
Maximum forced delay: 120

First 10 flight variables:


,flight_id,std,sta,forced_delay,c_cancel,c_delay_per_min,passengers,aircraft_type,origin,destination,cancel_lb,cancel_ub,cancel_integrality,delay_lb,delay_ub,delay_integrality
0,FL-1000,330,480,0,690440,665,166,A320,DEL,GOI,0,1,1,0,180,0
1,FL-1001,331,491,0,720475,785,175,A320,DEL,BLR,0,1,1,0,180,0
2,FL-1002,331,456,0,378492,930,65,ATR72,DEL,BOM,0,1,1,0,180,0
3,FL-1003,333,463,0,699701,719,148,A320,DEL,CCU,0,1,1,0,180,0
4,FL-1004,333,463,15,381809,732,61,ATR72,HYD,DEL,0,1,1,15,180,0
5,FL-1005,335,430,0,651167,758,149,A320,BOM,BLR,0,1,1,0,180,0
6,FL-1006,336,431,0,941475,746,203,A321,BOM,BLR,0,1,1,0,180,0
7,FL-1007,336,461,0,760626,777,157,A320,DEL,BOM,0,1,1,0,180,0
8,FL-1008,336,496,0,827880,853,184,A321,DEL,BLR,0,1,1,0,180,0
9,FL-1009,337,437,0,821527,696,174,A320,BLR,BOM,0,1,1,0,180,0


In [46]:
# PRESOLVER — CELL 23
# Create aircraft assignment variable indices

aircraft_variables = (
    aircraft_reduced_candidates[
        [
            "flight_id",
            "aircraft_id",
            "aircraft_type",
            "origin",
            "destination",
            "std",
            "sta",
            "passengers"
        ]
    ]
    .drop_duplicates(
        subset=["flight_id", "aircraft_id"]
    )
    .reset_index(drop=True)
)

# Unique MILP variable index
aircraft_variables["variable_id"] = range(
    len(aircraft_variables)
)

# Binary variable bounds
aircraft_variables["lb"] = 0
aircraft_variables["ub"] = 1
aircraft_variables["integrality"] = 1

print("Aircraft assignment variables:",
      len(aircraft_variables))

print("Variable IDs:",
      aircraft_variables["variable_id"].min(),
      "to",
      aircraft_variables["variable_id"].max())

print("\nBinary variables:",
      (aircraft_variables["integrality"] == 1).sum())

print("\nFirst 10 aircraft variables:")
display(aircraft_variables.head(10))


Aircraft assignment variables: 26488
Variable IDs: 0 to 26487

Binary variables: 26488

First 10 aircraft variables:


,flight_id,aircraft_id,aircraft_type,origin,destination,std,sta,passengers,variable_id,lb,ub,integrality
0,FL-1000,VT-AC102,A320,DEL,GOI,330,480,166,0,0,1,1
1,FL-1000,VT-AC103,A320,DEL,GOI,330,480,166,1,0,1,1
2,FL-1000,VT-AC104,A320,DEL,GOI,330,480,166,2,0,1,1
3,FL-1000,VT-AC108,A320,DEL,GOI,330,480,166,3,0,1,1
4,FL-1000,VT-AC110,A320,DEL,GOI,330,480,166,4,0,1,1
5,FL-1000,VT-AC111,A320,DEL,GOI,330,480,166,5,0,1,1
6,FL-1000,VT-AC113,A320,DEL,GOI,330,480,166,6,0,1,1
7,FL-1000,VT-AC114,A320,DEL,GOI,330,480,166,7,0,1,1
8,FL-1000,VT-AC117,A320,DEL,GOI,330,480,166,8,0,1,1
9,FL-1000,VT-AC120,A320,DEL,GOI,330,480,166,9,0,1,1


In [47]:
# PRESOLVER — CELL 24
# Create global variable indices for cancellation and delay

num_aircraft_vars = len(aircraft_variables)
num_flights = len(flight_variables)

# Cancellation variables
flight_variables["cancel_variable_id"] = [
    num_aircraft_vars + i
    for i in range(num_flights)
]

# Delay variables
flight_variables["delay_variable_id"] = [
    num_aircraft_vars + num_flights + i
    for i in range(num_flights)
]

# Total variables so far
total_variables = (
    num_aircraft_vars
    + num_flights
    + num_flights
)

print("Aircraft variables :", num_aircraft_vars)
print("Cancellation vars  :", num_flights)
print("Delay variables    :", num_flights)
print("-------------------------------")
print("Total variables    :", total_variables)

print("\nVariable index ranges:")

print(
    "Aircraft      :",
    aircraft_variables["variable_id"].min(),
    "to",
    aircraft_variables["variable_id"].max()
)

print(
    "Cancellation  :",
    flight_variables["cancel_variable_id"].min(),
    "to",
    flight_variables["cancel_variable_id"].max()
)

print(
    "Delay         :",
    flight_variables["delay_variable_id"].min(),
    "to",
    flight_variables["delay_variable_id"].max()
)

print("\nFirst 10 flight variable mappings:")

display(
    flight_variables[
        [
            "flight_id",
            "cancel_variable_id",
            "delay_variable_id"
        ]
    ].head(10)
)

Aircraft variables : 26488
Cancellation vars  : 789
Delay variables    : 789
-------------------------------
Total variables    : 28066

Variable index ranges:
Aircraft      : 0 to 26487
Cancellation  : 26488 to 27276
Delay         : 27277 to 28065

First 10 flight variable mappings:


,flight_id,cancel_variable_id,delay_variable_id
0,FL-1000,26488,27277
1,FL-1001,26489,27278
2,FL-1002,26490,27279
3,FL-1003,26491,27280
4,FL-1004,26492,27281
5,FL-1005,26493,27282
6,FL-1006,26494,27283
7,FL-1007,26495,27284
8,FL-1008,26496,27285
9,FL-1009,26497,27286


In [48]:
# PRESOLVER — CELL 25
# Create indexed crew assignment variables

crew_variables = (
    crew_candidates[
        [
            "flight_id",
            "crew_id",
            "crew_type",
            "type_rating"
        ]
    ]
    .drop_duplicates(
        subset=["flight_id", "crew_id"]
    )
    .reset_index(drop=True)
)

# Global variable IDs
crew_variables["variable_id"] = [
    total_variables + i
    for i in range(len(crew_variables))
]

# Binary variable bounds
crew_variables["lb"] = 0
crew_variables["ub"] = 1
crew_variables["integrality"] = 1

print("Crew assignment variables:",
      len(crew_variables))

print("\nCrew variables by role:")
print(
    crew_variables["crew_type"]
    .value_counts()
    .sort_index()
)

print("\nVariable ID range:",
      crew_variables["variable_id"].min(),
      "to",
      crew_variables["variable_id"].max()
)

print("\nFirst 10 crew variables:")
display(crew_variables.head(10))

Crew assignment variables: 106264

Crew variables by role:
crew_type
CABIN_CREW       41943
CAPTAIN          32888
FIRST_OFFICER    31433
Name: count, dtype: int64

Variable ID range: 28066 to 134329

First 10 crew variables:


,flight_id,crew_id,crew_type,type_rating,variable_id,lb,ub,integrality
0,FL-1000,CRW-0003,CAPTAIN,A320,28066,0,1,1
1,FL-1000,CRW-0006,CAPTAIN,A320,28067,0,1,1
2,FL-1000,CRW-0007,CAPTAIN,A320,28068,0,1,1
3,FL-1000,CRW-0008,CAPTAIN,A320,28069,0,1,1
4,FL-1000,CRW-0013,CAPTAIN,A320,28070,0,1,1
5,FL-1000,CRW-0014,CAPTAIN,A320,28071,0,1,1
6,FL-1000,CRW-0016,CAPTAIN,A320,28072,0,1,1
7,FL-1000,CRW-0017,CAPTAIN,A320,28073,0,1,1
8,FL-1000,CRW-0018,CAPTAIN,A320,28074,0,1,1
9,FL-1000,CRW-0019,CAPTAIN,A320,28075,0,1,1


In [49]:
# PRESOLVER — CELL 26
# Create indexed aircraft sequencing variables

aircraft_arc_variables = (
    aircraft_sequence_arcs[
        [
            "from_flight",
            "to_flight",
            "aircraft_id",
            "aircraft_type",
            "connection_gap"
        ]
    ]
    .drop_duplicates(
        subset=["from_flight", "to_flight", "aircraft_id"]
    )
    .reset_index(drop=True)
)

# Starting index after all existing variables
crew_end = crew_variables["variable_id"].max()

aircraft_arc_variables["variable_id"] = [
    crew_end + 1 + i
    for i in range(len(aircraft_arc_variables))
]

# Binary sequencing variables
aircraft_arc_variables["lb"] = 0
aircraft_arc_variables["ub"] = 1
aircraft_arc_variables["integrality"] = 1

# Total variable count
total_variables = (
    aircraft_arc_variables["variable_id"].max() + 1
)

print("Aircraft sequencing variables:",
      len(aircraft_arc_variables))

print("\nVariable ID range:",
      aircraft_arc_variables["variable_id"].min(),
      "to",
      aircraft_arc_variables["variable_id"].max())

print("\nTotal MILP variables so far:",
      total_variables)

print("\nFirst 10 sequencing variables:")
display(aircraft_arc_variables.head(10))

Aircraft sequencing variables: 338468

Variable ID range: 134330 to 472797

Total MILP variables so far: 472798

First 10 sequencing variables:


,from_flight,to_flight,aircraft_id,aircraft_type,connection_gap,variable_id,lb,ub,integrality
0,FL-1000,FL-1149,VT-AC102,A320,45,134330,0,1,1
1,FL-1000,FL-1149,VT-AC103,A320,45,134331,0,1,1
2,FL-1000,FL-1149,VT-AC104,A320,45,134332,0,1,1
3,FL-1000,FL-1149,VT-AC108,A320,45,134333,0,1,1
4,FL-1000,FL-1149,VT-AC110,A320,45,134334,0,1,1
5,FL-1000,FL-1149,VT-AC111,A320,45,134335,0,1,1
6,FL-1000,FL-1149,VT-AC113,A320,45,134336,0,1,1
7,FL-1000,FL-1149,VT-AC114,A320,45,134337,0,1,1
8,FL-1000,FL-1149,VT-AC117,A320,45,134338,0,1,1
9,FL-1000,FL-1149,VT-AC120,A320,45,134339,0,1,1


In [50]:
# PRESOLVER — CELL 27
# Analyze aircraft starting-position candidates

start_candidates = aircraft_variables[
    aircraft_variables["flight_id"].isin(
        initial_location_candidates["flight_id"]
    )
].merge(
    initial_location_candidates[
        ["flight_id", "aircraft_id"]
    ].drop_duplicates(),
    on=["flight_id", "aircraft_id"],
    how="inner"
)

print("Aircraft assignment variables:",
      len(aircraft_variables))

print("Valid initial-position assignments:",
      len(start_candidates))

print(
    "Aircraft assignments requiring a predecessor:",
    len(aircraft_variables) - len(start_candidates)
)

print(
    "\nFlights with at least one valid initial aircraft:",
    start_candidates["flight_id"].nunique()
)

print(
    "Aircraft with at least one valid initial flight:",
    start_candidates["aircraft_id"].nunique()
)

print("\nInitial candidates by aircraft type:")
print(
    start_candidates["aircraft_type"]
    .value_counts()
    .sort_index()
)

print("\nFirst 10 initial-position candidates:")
display(start_candidates.head(10))

Aircraft assignment variables: 26488
Valid initial-position assignments: 5501
Aircraft assignments requiring a predecessor: 20987

Flights with at least one valid initial aircraft: 547
Aircraft with at least one valid initial flight: 116

Initial candidates by aircraft type:
aircraft_type
A320     3945
A321      970
ATR72     130
B737      456
Name: count, dtype: int64

First 10 initial-position candidates:


,flight_id,aircraft_id,aircraft_type,origin,destination,std,sta,passengers,variable_id,lb,ub,integrality
0,FL-1000,VT-AC102,A320,DEL,GOI,330,480,166,0,0,1,1
1,FL-1000,VT-AC103,A320,DEL,GOI,330,480,166,1,0,1,1
2,FL-1000,VT-AC104,A320,DEL,GOI,330,480,166,2,0,1,1
3,FL-1000,VT-AC108,A320,DEL,GOI,330,480,166,3,0,1,1
4,FL-1000,VT-AC110,A320,DEL,GOI,330,480,166,4,0,1,1
5,FL-1000,VT-AC111,A320,DEL,GOI,330,480,166,5,0,1,1
6,FL-1000,VT-AC113,A320,DEL,GOI,330,480,166,6,0,1,1
7,FL-1000,VT-AC114,A320,DEL,GOI,330,480,166,7,0,1,1
8,FL-1000,VT-AC117,A320,DEL,GOI,330,480,166,8,0,1,1
9,FL-1000,VT-AC120,A320,DEL,GOI,330,480,166,9,0,1,1


In [51]:
# PRESOLVER — CELL 28
# Build flight coverage constraint structure

flight_coverage_constraints = []

# Group aircraft candidates by flight
aircraft_by_flight = (
    aircraft_variables
    .groupby("flight_id")
)

for _, flight in flight_variables.iterrows():

    flight_id = flight["flight_id"]

    # Aircraft variables available for this flight
    if flight_id in aircraft_by_flight.groups:

        candidates = aircraft_by_flight.get_group(flight_id)

        aircraft_variable_ids = (
            candidates["variable_id"].tolist()
        )

    else:
        aircraft_variable_ids = []

    # Cancellation variable
    cancel_variable_id = flight["cancel_variable_id"]

    # Store constraint
    flight_coverage_constraints.append({
        "flight_id": flight_id,
        "aircraft_variable_ids": aircraft_variable_ids,
        "cancel_variable_id": cancel_variable_id,
        "rhs": 1
    })

print(
    "Flight coverage constraints:",
    len(flight_coverage_constraints)
)

print(
    "Flights represented:",
    len(flight_coverage_constraints)
)

print(
    "Flights with no aircraft candidates:",
    sum(
        len(c["aircraft_variable_ids"]) == 0
        for c in flight_coverage_constraints
    )
)

print("\nExample constraint:")

example = flight_coverage_constraints[0]

print(
    example["flight_id"],
    ":",
    len(example["aircraft_variable_ids"]),
    "aircraft variables + cancellation variable",
    example["cancel_variable_id"],
    "= 1"
)

Flight coverage constraints: 789
Flights represented: 789
Flights with no aircraft candidates: 0

Example constraint:
FL-1000 : 29 aircraft variables + cancellation variable 26488 = 1


In [52]:
# PRESOLVER — CELL 29
# Build incoming/outgoing arc counts for each aircraft-flight candidate

# Count incoming sequencing arcs
incoming_counts = (
    aircraft_arc_variables
    .groupby(["aircraft_id", "to_flight"])
    .size()
    .reset_index(name="incoming_arc_count")
)

# Count outgoing sequencing arcs
outgoing_counts = (
    aircraft_arc_variables
    .groupby(["aircraft_id", "from_flight"])
    .size()
    .reset_index(name="outgoing_arc_count")
)

# Start-position pairs
initial_pairs = set(
    zip(
        initial_location_candidates["flight_id"],
        initial_location_candidates["aircraft_id"]
    )
)

# Start with aircraft variables
aircraft_flow_candidates = aircraft_variables[
    [
        "flight_id",
        "aircraft_id",
        "aircraft_type"
    ]
].copy()

# Reconstruct can_start
aircraft_flow_candidates["can_start"] = [
    (f, a) in initial_pairs
    for f, a in zip(
        aircraft_flow_candidates["flight_id"],
        aircraft_flow_candidates["aircraft_id"]
    )
]

# Add incoming arc counts
aircraft_flow_candidates = aircraft_flow_candidates.merge(
    incoming_counts,
    left_on=["aircraft_id", "flight_id"],
    right_on=["aircraft_id", "to_flight"],
    how="left"
)

# Add outgoing arc counts
aircraft_flow_candidates = aircraft_flow_candidates.merge(
    outgoing_counts,
    left_on=["aircraft_id", "flight_id"],
    right_on=["aircraft_id", "from_flight"],
    how="left"
)

# Remove helper columns created by merges
aircraft_flow_candidates = aircraft_flow_candidates.drop(
    columns=["to_flight", "from_flight"],
    errors="ignore"
)

# Missing means no arc
aircraft_flow_candidates["incoming_arc_count"] = (
    aircraft_flow_candidates["incoming_arc_count"]
    .fillna(0)
    .astype(int)
)

aircraft_flow_candidates["outgoing_arc_count"] = (
    aircraft_flow_candidates["outgoing_arc_count"]
    .fillna(0)
    .astype(int)
)

# A candidate has an entry if:
# 1. aircraft can start at its initial location, OR
# 2. aircraft has a feasible predecessor
aircraft_flow_candidates["has_entry"] = (
    aircraft_flow_candidates["can_start"]
    |
    (aircraft_flow_candidates["incoming_arc_count"] > 0)
)

# No outgoing arc is NOT automatically infeasible:
# the flight may simply be the aircraft's final flight.
aircraft_flow_candidates["has_exit"] = (
    aircraft_flow_candidates["outgoing_arc_count"] > 0
)

print("Aircraft-flight candidates:",
      len(aircraft_flow_candidates))

print("\nCandidates with no valid entry:",
      (~aircraft_flow_candidates["has_entry"]).sum())

print("Candidates with no outgoing arc:",
      (~aircraft_flow_candidates["has_exit"]).sum())

print("\nIncoming arc statistics:")
print(
    aircraft_flow_candidates["incoming_arc_count"].describe()
)

print("\nOutgoing arc statistics:")
print(
    aircraft_flow_candidates["outgoing_arc_count"].describe()
)

print("\nFirst 10 candidates:")
display(aircraft_flow_candidates.head(10))

Aircraft-flight candidates: 26488

Candidates with no valid entry: 0
Candidates with no outgoing arc: 4570

Incoming arc statistics:
count    26488.000000
mean        12.778164
std         12.737628
min          0.000000
25%          3.000000
50%          9.000000
75%         18.000000
max         59.000000
Name: incoming_arc_count, dtype: float64

Outgoing arc statistics:
count    26488.000000
mean        12.778164
std         13.451681
min          0.000000
25%          2.000000
50%          9.000000
75%         18.000000
max         69.000000
Name: outgoing_arc_count, dtype: float64

First 10 candidates:


,flight_id,aircraft_id,aircraft_type,can_start,incoming_arc_count,outgoing_arc_count,has_entry,has_exit
0,FL-1000,VT-AC102,A320,True,0,22,True,True
1,FL-1000,VT-AC103,A320,True,0,22,True,True
2,FL-1000,VT-AC104,A320,True,0,22,True,True
3,FL-1000,VT-AC108,A320,True,0,22,True,True
4,FL-1000,VT-AC110,A320,True,0,22,True,True
5,FL-1000,VT-AC111,A320,True,0,22,True,True
6,FL-1000,VT-AC113,A320,True,0,22,True,True
7,FL-1000,VT-AC114,A320,True,0,22,True,True
8,FL-1000,VT-AC117,A320,True,0,22,True,True
9,FL-1000,VT-AC120,A320,True,0,22,True,True


In [53]:
# PRESOLVER — CELL 30
# Build aircraft flow constraint structure (optimized)

# Create fast lookup dictionaries for incoming/outgoing arcs
incoming_lookup = (
    aircraft_arc_variables
    .groupby(["aircraft_id", "to_flight"])["variable_id"]
    .apply(list)
    .to_dict()
)

outgoing_lookup = (
    aircraft_arc_variables
    .groupby(["aircraft_id", "from_flight"])["variable_id"]
    .apply(list)
    .to_dict()
)

# Fast lookup for aircraft assignment variable IDs
aircraft_variable_lookup = (
    aircraft_variables
    .set_index(["aircraft_id", "flight_id"])["variable_id"]
    .to_dict()
)

aircraft_flow_constraints = []

for _, row in aircraft_flow_candidates.iterrows():

    flight_id = row["flight_id"]
    aircraft_id = row["aircraft_id"]

    # Aircraft assignment variable x[f,a]
    x_variable_id = aircraft_variable_lookup[
        (aircraft_id, flight_id)
    ]

    # Incoming sequencing variables
    incoming_variable_ids = incoming_lookup.get(
        (aircraft_id, flight_id),
        []
    )

    # Outgoing sequencing variables
    outgoing_variable_ids = outgoing_lookup.get(
        (aircraft_id, flight_id),
        []
    )

    aircraft_flow_constraints.append({
        "flight_id": flight_id,
        "aircraft_id": aircraft_id,
        "aircraft_variable_id": x_variable_id,
        "incoming_arc_variable_ids": incoming_variable_ids,
        "outgoing_arc_variable_ids": outgoing_variable_ids,
        "can_start": row["can_start"]
    })

print(
    "Aircraft flow constraints:",
    len(aircraft_flow_constraints)
)

print(
    "Expected aircraft-flight constraints:",
    len(aircraft_variables)
)

print(
    "Constraints with valid entry:",
    sum(
        c["can_start"] or len(c["incoming_arc_variable_ids"]) > 0
        for c in aircraft_flow_constraints
    )
)

print(
    "Constraints with no outgoing arc:",
    sum(
        len(c["outgoing_arc_variable_ids"]) == 0
        for c in aircraft_flow_constraints
    )
)

print("\nExample constraint:")

example = aircraft_flow_constraints[0]

print(
    "Flight:", example["flight_id"],
    "| Aircraft:", example["aircraft_id"]
)

print(
    "Aircraft variable:",
    example["aircraft_variable_id"]
)

print(
    "Incoming arcs:",
    len(example["incoming_arc_variable_ids"])
)

print(
    "Outgoing arcs:",
    len(example["outgoing_arc_variable_ids"])
)

print(
    "Can start:",
    example["can_start"]
)

Aircraft flow constraints: 26488
Expected aircraft-flight constraints: 26488
Constraints with valid entry: 26488
Constraints with no outgoing arc: 4570

Example constraint:
Flight: FL-1000 | Aircraft: VT-AC102
Aircraft variable: 0
Incoming arcs: 0
Outgoing arcs: 22
Can start: True


In [54]:
# PRESOLVER — CELL 31
# Validate aircraft path boundary structure

aircraft_flow_candidates["has_incoming"] = (
    aircraft_flow_candidates["incoming_arc_count"] > 0
)

aircraft_flow_candidates["has_outgoing"] = (
    aircraft_flow_candidates["outgoing_arc_count"] > 0
)

print("Total aircraft-flight candidates:",
      len(aircraft_flow_candidates))

print("\nStartable candidates:",
      aircraft_flow_candidates["can_start"].sum())

print("Candidates requiring a predecessor:",
      (
          ~aircraft_flow_candidates["can_start"]
      ).sum())

print("\nCandidates with incoming arcs:",
      aircraft_flow_candidates["has_incoming"].sum())

print("Candidates with no incoming arcs:",
      (
          ~aircraft_flow_candidates["has_incoming"]
      ).sum())

print("\nCandidates with outgoing arcs:",
      aircraft_flow_candidates["has_outgoing"].sum())

print("Terminal candidates:",
      (
          ~aircraft_flow_candidates["has_outgoing"]
      ).sum())

print("\nStartable AND terminal:",
      (
          aircraft_flow_candidates["can_start"]
          &
          ~aircraft_flow_candidates["has_outgoing"]
      ).sum())

print("\nPredecessor-required AND terminal:",
      (
          ~aircraft_flow_candidates["can_start"]
          &
          ~aircraft_flow_candidates["has_outgoing"]
      ).sum())

print("\nSanity check — candidates with no entry:",
      (
          ~aircraft_flow_candidates["has_entry"]
      ).sum())

Total aircraft-flight candidates: 26488

Startable candidates: 5501
Candidates requiring a predecessor: 20987

Candidates with incoming arcs: 24318
Candidates with no incoming arcs: 2170

Candidates with outgoing arcs: 21918
Terminal candidates: 4570

Startable AND terminal: 743

Predecessor-required AND terminal: 3827

Sanity check — candidates with no entry: 0


In [55]:
# PRESOLVER — CELL 32
# Build aircraft flow constraint structure

aircraft_flow_constraint_rows = []

for c in aircraft_flow_constraints:

    x_id = c["aircraft_variable_id"]
    incoming_ids = c["incoming_arc_variable_ids"]
    can_start = c["can_start"]

    # Constraint:
    # incoming arcs + start condition = aircraft assignment
    #
    # If can_start:
    #     x - incoming_arcs <= 1
    #
    # If cannot start:
    #     x - incoming_arcs = 0

    if can_start:
        sense = "<="
        rhs = 1
    else:
        sense = "="
        rhs = 0

    aircraft_flow_constraint_rows.append({
        "flight_id": c["flight_id"],
        "aircraft_id": c["aircraft_id"],
        "aircraft_variable_id": x_id,
        "incoming_arc_variable_ids": incoming_ids,
        "sense": sense,
        "rhs": rhs
    })

print(
    "Aircraft flow constraints:",
    len(aircraft_flow_constraint_rows)
)

print(
    "Equality constraints:",
    sum(
        r["sense"] == "="
        for r in aircraft_flow_constraint_rows
    )
)

print(
    "Inequality constraints:",
    sum(
        r["sense"] == "<="
        for r in aircraft_flow_constraint_rows
    )
)

print("\nExample:")
print(aircraft_flow_constraint_rows[0])

Aircraft flow constraints: 26488
Equality constraints: 20987
Inequality constraints: 5501

Example:
{'flight_id': 'FL-1000', 'aircraft_id': 'VT-AC102', 'aircraft_variable_id': 0, 'incoming_arc_variable_ids': [], 'sense': '<=', 'rhs': 1}


In [56]:
# PRESOLVER — CELL 33
# Build outgoing aircraft flow constraints

aircraft_outgoing_constraint_rows = []

for c in aircraft_flow_constraints:

    x_id = c["aircraft_variable_id"]
    outgoing_ids = c["outgoing_arc_variable_ids"]

    # x - outgoing arcs <= 0
    # This ensures that if a flight is selected,
    # the aircraft can continue to a successor OR terminate.

    aircraft_outgoing_constraint_rows.append({
        "flight_id": c["flight_id"],
        "aircraft_id": c["aircraft_id"],
        "aircraft_variable_id": x_id,
        "outgoing_arc_variable_ids": outgoing_ids,
        "sense": "<=",
        "rhs": 0
    })

print(
    "Outgoing flow constraints:",
    len(aircraft_outgoing_constraint_rows)
)

print(
    "Constraints with outgoing arcs:",
    sum(
        len(r["outgoing_arc_variable_ids"]) > 0
        for r in aircraft_outgoing_constraint_rows
    )
)

print(
    "Terminal constraints:",
    sum(
        len(r["outgoing_arc_variable_ids"]) == 0
        for r in aircraft_outgoing_constraint_rows
    )
)

print("\nExample:")
print(aircraft_outgoing_constraint_rows[0])

Outgoing flow constraints: 26488
Constraints with outgoing arcs: 21918
Terminal constraints: 4570

Example:
{'flight_id': 'FL-1000', 'aircraft_id': 'VT-AC102', 'aircraft_variable_id': 0, 'outgoing_arc_variable_ids': [134330, 134359, 134388, 134417, 134446, 134475, 134504, 134533, 134562, 134591, 134620, 134649, 134678, 134707, 134736, 134765, 134794, 134823, 134852, 134881, 134910, 134939], 'sense': '<=', 'rhs': 0}


In [57]:
# PRESOLVER — CELL 34
# Combine aircraft flow constraints

all_aircraft_flow_constraints = (
    aircraft_flow_constraint_rows
    + aircraft_outgoing_constraint_rows
)

print(
    "Total aircraft flow constraints:",
    len(all_aircraft_flow_constraints)
)

print(
    "Incoming flow constraints:",
    len(aircraft_flow_constraint_rows)
)

print(
    "Outgoing flow constraints:",
    len(aircraft_outgoing_constraint_rows)
)

print(
    "Expected total:",
    2 * len(aircraft_variables)
)

print("\nSanity check:",
      len(all_aircraft_flow_constraints)
      == 2 * len(aircraft_variables))

Total aircraft flow constraints: 52976
Incoming flow constraints: 26488
Outgoing flow constraints: 26488
Expected total: 52976

Sanity check: True


In [58]:
# PRESOLVER — CELL 35
# Build delay/cancellation constraints

delay_cancel_constraints = []

for _, flight in flight_variables.iterrows():

    flight_id = flight["flight_id"]
    cancel_id = flight["cancel_variable_id"]
    delay_id = flight["delay_variable_id"]
    forced_delay = flight["forced_delay"]

    # Delay must be zero if the flight is cancelled:
    #
    # delay + M * cancel <= M
    #
    # Using M = 180 (current delay upper bound)

    M = 180

    delay_cancel_constraints.append({
        "flight_id": flight_id,
        "delay_variable_id": delay_id,
        "cancel_variable_id": cancel_id,
        "sense": "<=",
        "rhs": M,
        "big_m": M
    })

    # If flight operates, delay must be at least forced_delay.
    #
    # delay + forced_delay * cancel >= forced_delay

    delay_cancel_constraints.append({
        "flight_id": flight_id,
        "delay_variable_id": delay_id,
        "cancel_variable_id": cancel_id,
        "sense": ">=",
        "rhs": forced_delay,
        "forced_delay": forced_delay
    })

print(
    "Delay/cancellation constraints:",
    len(delay_cancel_constraints)
)

print(
    "Flights represented:",
    len(
        set(c["flight_id"] for c in delay_cancel_constraints)
    )
)

print(
    "Flights with forced delay:",
    (flight_variables["forced_delay"] > 0).sum()
)

print("\nExample:")
print(delay_cancel_constraints[0])
print(delay_cancel_constraints[1])

Delay/cancellation constraints: 1578
Flights represented: 789
Flights with forced delay: 60

Example:
{'flight_id': 'FL-1000', 'delay_variable_id': 27277, 'cancel_variable_id': 26488, 'sense': '<=', 'rhs': 180, 'big_m': 180}
{'flight_id': 'FL-1000', 'delay_variable_id': 27277, 'cancel_variable_id': 26488, 'sense': '>=', 'rhs': 0, 'forced_delay': 0}


In [59]:
# PRESOLVER — CELL 36
# Build objective coefficients for flight variables

flight_objective = []

for _, flight in flight_variables.iterrows():

    flight_objective.append({
        "flight_id": flight["flight_id"],
        "cancel_variable_id": flight["cancel_variable_id"],
        "cancel_cost": flight["c_cancel"],
        "delay_variable_id": flight["delay_variable_id"],
        "delay_cost_per_min": flight["c_delay_per_min"]
    })

print(
    "Flights in objective:",
    len(flight_objective)
)

print(
    "Cancellation variables:",
    len(
        [x for x in flight_objective
         if x["cancel_variable_id"] is not None]
    )
)

print(
    "Delay variables:",
    len(
        [x for x in flight_objective
         if x["delay_variable_id"] is not None]
    )
)

print("\nExample:")
print(flight_objective[0])

Flights in objective: 789
Cancellation variables: 789
Delay variables: 789

Example:
{'flight_id': 'FL-1000', 'cancel_variable_id': 26488, 'cancel_cost': 690440, 'delay_variable_id': 27277, 'delay_cost_per_min': 665}


In [60]:
# PRESOLVER — CELL 37
# Build objective vector for all current variables

import numpy as np

# Start with zero objective coefficients
objective_coefficients = np.zeros(total_variables)

# Cancellation + delay costs
for _, flight in flight_variables.iterrows():

    objective_coefficients[
        flight["cancel_variable_id"]
    ] = flight["c_cancel"]

    objective_coefficients[
        flight["delay_variable_id"]
    ] = flight["c_delay_per_min"]

# Aircraft assignment and sequencing variables currently have zero cost
# because the dataset does not provide an aircraft/connection cost.

print("Total variables:", total_variables)

print(
    "Objective vector length:",
    len(objective_coefficients)
)

print(
    "Non-zero objective coefficients:",
    np.count_nonzero(objective_coefficients)
)

print(
    "Zero objective coefficients:",
    np.sum(objective_coefficients == 0)
)

print("\nExample coefficients:")
print(
    "FL-1000 cancellation:",
    objective_coefficients[
        flight_variables.iloc[0]["cancel_variable_id"]
    ]
)

print(
    "FL-1000 delay:",
    objective_coefficients[
        flight_variables.iloc[0]["delay_variable_id"]
    ]
)

Total variables: 472798
Objective vector length: 472798
Non-zero objective coefficients: 1578
Zero objective coefficients: 471220

Example coefficients:
FL-1000 cancellation: 690440.0
FL-1000 delay: 665.0


In [61]:
# PRESOLVER — CELL 38
# Constraint count summary

num_flight_coverage = len(flight_coverage_constraints)
num_aircraft_flow = len(all_aircraft_flow_constraints)
num_delay_cancel = len(delay_cancel_constraints)

total_constraints = (
    num_flight_coverage
    + num_aircraft_flow
    + num_delay_cancel
)

print("Constraint summary")
print("------------------")
print("Flight coverage constraints:", num_flight_coverage)
print("Aircraft flow constraints:", num_aircraft_flow)
print("Delay/cancellation constraints:", num_delay_cancel)
print("Total constraints:", total_constraints)

print("\nVariable summary")
print("----------------")
print("Aircraft assignment variables:", len(aircraft_variables))
print("Cancellation variables:", len(flight_variables))
print("Delay variables:", len(flight_variables))
print("Aircraft sequencing variables:", len(aircraft_arc_variables))
print("Total variables:", total_variables)

Constraint summary
------------------
Flight coverage constraints: 789
Aircraft flow constraints: 52976
Delay/cancellation constraints: 1578
Total constraints: 55343

Variable summary
----------------
Aircraft assignment variables: 26488
Cancellation variables: 789
Delay variables: 789
Aircraft sequencing variables: 338468
Total variables: 472798


In [62]:
# PRESOLVER — CELL 39
# Build crew coverage constraint structure

crew_coverage_constraints = []

for _, flight in flight_variables.iterrows():

    flight_id = flight["flight_id"]

    for role in ["CAPTAIN", "FIRST_OFFICER", "CABIN_CREW"]:

        role_candidates = crew_candidate_sets[role]

        candidate_ids = crew_variables.loc[
            (
                crew_variables["flight_id"] == flight_id
            ) &
            (
                crew_variables["crew_type"] == role
            ),
            "variable_id"
        ].tolist()

        crew_coverage_constraints.append({
            "flight_id": flight_id,
            "crew_role": role,
            "crew_variable_ids": candidate_ids,
            "cancel_variable_id": flight["cancel_variable_id"],
            "rhs": 1
        })

print(
    "Crew coverage constraints:",
    len(crew_coverage_constraints)
)

print(
    "Expected:",
    len(flight_variables) * 3
)

print(
    "Flights represented:",
    len(
        set(c["flight_id"] for c in crew_coverage_constraints)
    )
)

print(
    "Constraints with no crew candidates:",
    sum(
        len(c["crew_variable_ids"]) == 0
        for c in crew_coverage_constraints
    )
)

print("\nConstraints by role:")

for role in ["CAPTAIN", "FIRST_OFFICER", "CABIN_CREW"]:
    print(
        role + ":",
        sum(c["crew_role"] == role
            for c in crew_coverage_constraints)
    )

print("\nExample:")
print(crew_coverage_constraints[0])

Crew coverage constraints: 2367
Expected: 2367
Flights represented: 789
Constraints with no crew candidates: 0

Constraints by role:
CAPTAIN: 789
FIRST_OFFICER: 789
CABIN_CREW: 789

Example:
{'flight_id': 'FL-1000', 'crew_role': 'CAPTAIN', 'crew_variable_ids': [28066, 28067, 28068, 28069, 28070, 28071, 28072, 28073, 28074, 28075, 28076, 28077, 28078, 28079, 28080, 28081, 28082, 28083, 28084, 28085, 28086, 28087, 28088, 28089, 28090, 28091, 28092, 28093, 28094, 28095, 28096, 28097, 28098, 28099, 28100, 28101, 28102, 28103, 28104, 28105, 28106, 28107, 28108, 28109, 28110, 28111, 28112, 28113, 28114, 28115, 28116, 28117, 28118, 28119, 28120, 28121, 28122, 28123, 28124], 'cancel_variable_id': 26488, 'rhs': 1}


In [63]:
# Correct aircraft incoming/outgoing flow constraints

aircraft_flow_constraint_rows = []
aircraft_outgoing_constraint_rows = []

for c in aircraft_flow_constraints:

    x_id = c["aircraft_variable_id"]
    incoming_ids = c["incoming_arc_variable_ids"]
    outgoing_ids = c["outgoing_arc_variable_ids"]
    can_start = c["can_start"]

    # Incoming:
    # sum(incoming arcs) <= x
    aircraft_flow_constraint_rows.append({
        "flight_id": c["flight_id"],
        "aircraft_id": c["aircraft_id"],
        "aircraft_variable_id": x_id,
        "incoming_arc_variable_ids": incoming_ids,
        "can_start": can_start,
        "sense": "<=",
        "rhs": 0
    })

    # If aircraft cannot start here, a selected flight MUST have a predecessor
    if not can_start:
        aircraft_flow_constraint_rows.append({
            "flight_id": c["flight_id"],
            "aircraft_id": c["aircraft_id"],
            "aircraft_variable_id": x_id,
            "incoming_arc_variable_ids": incoming_ids,
            "can_start": can_start,
            "sense": ">=",
            "rhs": 0
        })

    # Outgoing:
    # sum(outgoing arcs) <= x
    aircraft_outgoing_constraint_rows.append({
        "flight_id": c["flight_id"],
        "aircraft_id": c["aircraft_id"],
        "aircraft_variable_id": x_id,
        "outgoing_arc_variable_ids": outgoing_ids,
        "sense": "<=",
        "rhs": 0
    })

print("Incoming constraint rows:", len(aircraft_flow_constraint_rows))
print("Outgoing constraint rows:", len(aircraft_outgoing_constraint_rows))

Incoming constraint rows: 47475
Outgoing constraint rows: 26488


In [64]:
# Build coefficient rows for aircraft flow constraints

aircraft_flow_rows = []

for c in aircraft_flow_constraints:

    x_id = c["aircraft_variable_id"]
    incoming_ids = c["incoming_arc_variable_ids"]
    can_start = c["can_start"]

    # sum(incoming) - x <= 0
    aircraft_flow_rows.append({
        "flight_id": c["flight_id"],
        "aircraft_id": c["aircraft_id"],
        "variable_ids": incoming_ids + [x_id],
        "coefficients": [1] * len(incoming_ids) + [-1],
        "sense": "<=",
        "rhs": 0
    })

    # For flights that cannot start directly:
    # x - sum(incoming) <= 0
    if not can_start:
        aircraft_flow_rows.append({
            "flight_id": c["flight_id"],
            "aircraft_id": c["aircraft_id"],
            "variable_ids": [x_id] + incoming_ids,
            "coefficients": [1] + [-1] * len(incoming_ids),
            "sense": "<=",
            "rhs": 0
        })

# Outgoing: sum(outgoing) - x <= 0
aircraft_outgoing_rows = []

for c in aircraft_flow_constraints:

    x_id = c["aircraft_variable_id"]
    outgoing_ids = c["outgoing_arc_variable_ids"]

    aircraft_outgoing_rows.append({
        "flight_id": c["flight_id"],
        "aircraft_id": c["aircraft_id"],
        "variable_ids": outgoing_ids + [x_id],
        "coefficients": [1] * len(outgoing_ids) + [-1],
        "sense": "<=",
        "rhs": 0
    })

print("Aircraft incoming rows:", len(aircraft_flow_rows))
print("Aircraft outgoing rows:", len(aircraft_outgoing_rows))
print("Total aircraft flow rows:",
      len(aircraft_flow_rows) + len(aircraft_outgoing_rows))

Aircraft incoming rows: 47475
Aircraft outgoing rows: 26488
Total aircraft flow rows: 73963


In [65]:
print(aircraft_flow_rows[0])
print(aircraft_flow_rows[-1])
print(aircraft_outgoing_rows[0])

{'flight_id': 'FL-1000', 'aircraft_id': 'VT-AC102', 'variable_ids': [0], 'coefficients': [-1], 'sense': '<=', 'rhs': 0}
{'flight_id': 'FL-1788', 'aircraft_id': 'VT-AC160', 'variable_ids': [26487, 136852, 149423, 158070, 174495, 176148, 179247, 182321, 198681, 222622, 225378, 243193, 269033, 271647, 274770, 277166, 313103, 322267, 331602, 333654, 336786, 340595, 345082, 363101, 376746, 385479, 392009, 402101, 405902, 408441, 414783, 416505, 421043, 424441, 426265, 427927, 436882, 438711, 442267, 444205, 445288, 447544, 449931, 453716, 454799, 456585, 457839, 459161, 460251, 462092, 462930, 463474, 465793, 467148, 467616, 467932, 469988, 470675, 472569, 472797], 'coefficients': [1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1], 'sense': '<=', 'rhs': 0}
{'flight_id': 'FL-1000', 'aircraft_id': 'VT-AC102

In [66]:
print("Crew connections:", len(crew_connections))
print(crew_connections.head())

Crew connections: 28679
  flight_1 flight_2 flight_1_type flight_2_type destination_1 origin_2  gap
0  FL-1000  FL-1149          A320          A320           GOI      GOI   45
1  FL-1000  FL-1165          A320          A321           GOI      GOI   58
2  FL-1000  FL-1210          A320          B737           GOI      GOI  107
3  FL-1000  FL-1217          A320          B737           GOI      GOI  114
4  FL-1000  FL-1227          A320          A320           GOI      GOI  132


In [67]:
print("Total overlapping pairs:", len(crew_conflicts))

# Check how many flight pairs have a gap < 45 minutes
print(
    "Crew connections with gap >= 45:",
    len(crew_connections)
)

Total overlapping pairs: 61120
Crew connections with gap >= 45: 28679


In [68]:
print(crew[["max_duty_minutes", "max_flights"]].drop_duplicates())

max_duty_minutes    480
max_flights           4
Name: 393, dtype: object


In [69]:
crew_assignment_counts = (
    crew_variables
    .groupby("crew_id")
    .size()
)

print("Crew members:", len(crew_assignment_counts))
print("Average candidate assignments:", crew_assignment_counts.mean())
print("Maximum candidate assignments:", crew_assignment_counts.max())
print("Crew members with >4 candidates:", (crew_assignment_counts > 4).sum())

Crew members: 388
Average candidate assignments: 273.87628865979383
Maximum candidate assignments: 389
Crew members with >4 candidates: 388


In [70]:
print(airport_capacity)


  airport  normal_hourly_capacity  fog_window_hourly_capacity  \
0     DEL                      40                          10   
1     BOM                      40                          35   
2     BLR                      40                          35   
3     HYD                      40                          35   
4     CCU                      40                          35   
5     MAA                      40                          35   
6     AMD                      40                          35   
7     PNQ                      40                          35   
8     GOI                      40                          35   
9     COK                      40                          35   

   disruption_start_min  disruption_end_min  
0                 360.0               570.0  
1                   NaN                 NaN  
2                   NaN                 NaN  
3                   NaN                 NaN  
4                   NaN                 NaN  
5       

In [71]:
# Create hourly airport movement candidates

airport_capacity_candidates = []

for _, flight in flights.iterrows():

    std = flight["std"]
    sta = flight["sta"]

    # Departure movement
    airport_capacity_candidates.append({
        "flight_id": flight["flight_id"],
        "airport": flight["origin"],
        "time_min": std,
        "movement_type": "departure"
    })

    # Arrival movement
    airport_capacity_candidates.append({
        "flight_id": flight["flight_id"],
        "airport": flight["destination"],
        "time_min": sta,
        "movement_type": "arrival"
    })

airport_capacity_candidates = pd.DataFrame(
    airport_capacity_candidates
)

airport_capacity_candidates["hour"] = (
    airport_capacity_candidates["time_min"] // 60
).astype(int)

print("Airport movement candidates:", len(airport_capacity_candidates))
print("Unique airports:", airport_capacity_candidates["airport"].nunique())
print("Unique hourly bins:", airport_capacity_candidates["hour"].nunique())
print(airport_capacity_candidates.head())

Airport movement candidates: 1578
Unique airports: 10
Unique hourly bins: 21
  flight_id airport  time_min movement_type  hour
0   FL-1000     DEL       330     departure     5
1   FL-1000     GOI       480       arrival     8
2   FL-1001     DEL       331     departure     5
3   FL-1001     BLR       491       arrival     8
4   FL-1002     DEL       331     departure     5


In [72]:
airport_capacity_constraints = []

capacity_lookup = airport_capacity.set_index("airport").to_dict("index")

for (airport, hour), group in airport_capacity_candidates.groupby(
    ["airport", "hour"]
):

    capacity_info = capacity_lookup[airport]

    # Hour interval in minutes
    hour_start = hour * 60
    hour_end = hour_start + 60

    # Use fog capacity if this hour overlaps DEL's disruption window
    if (
        pd.notna(capacity_info["disruption_start_min"])
        and hour_end > capacity_info["disruption_start_min"]
        and hour_start < capacity_info["disruption_end_min"]
    ):
        capacity = capacity_info["fog_window_hourly_capacity"]
    else:
        capacity = capacity_info["normal_hourly_capacity"]

    airport_capacity_constraints.append({
        "airport": airport,
        "hour": hour,
        "flight_ids": group["flight_id"].unique().tolist(),
        "capacity": int(capacity)
    })

print("Airport capacity constraints:", len(airport_capacity_constraints))
print("Normal-capacity constraints:",
      sum(c["capacity"] == 40 for c in airport_capacity_constraints))
print("Fog-capacity constraints:",
      sum(c["capacity"] == 10 for c in airport_capacity_constraints))

print("\nExample:")
print(airport_capacity_constraints[:3])

Airport capacity constraints: 194
Normal-capacity constraints: 190
Fog-capacity constraints: 4

Example:
[{'airport': 'AMD', 'hour': 7, 'flight_ids': ['FL-1020', 'FL-1035', 'FL-1067'], 'capacity': 40}, {'airport': 'AMD', 'hour': 8, 'flight_ids': ['FL-1043', 'FL-1085', 'FL-1086', 'FL-1092', 'FL-1115', 'FL-1124', 'FL-1138', 'FL-1147', 'FL-1152', 'FL-1160', 'FL-1162'], 'capacity': 40}, {'airport': 'AMD', 'hour': 9, 'flight_ids': ['FL-1117', 'FL-1181', 'FL-1184'], 'capacity': 40}]


In [73]:
print(airport_capacity_constraints[:5])

del_fog = [
    c for c in airport_capacity_constraints
    if c["capacity"] == 10
]

print("\nDEL fog constraints:")
print(del_fog)

[{'airport': 'AMD', 'hour': 7, 'flight_ids': ['FL-1020', 'FL-1035', 'FL-1067'], 'capacity': 40}, {'airport': 'AMD', 'hour': 8, 'flight_ids': ['FL-1043', 'FL-1085', 'FL-1086', 'FL-1092', 'FL-1115', 'FL-1124', 'FL-1138', 'FL-1147', 'FL-1152', 'FL-1160', 'FL-1162'], 'capacity': 40}, {'airport': 'AMD', 'hour': 9, 'flight_ids': ['FL-1117', 'FL-1181', 'FL-1184'], 'capacity': 40}, {'airport': 'AMD', 'hour': 10, 'flight_ids': ['FL-1121', 'FL-1130', 'FL-1145', 'FL-1164', 'FL-1191', 'FL-1226', 'FL-1248', 'FL-1249', 'FL-1255'], 'capacity': 40}, {'airport': 'AMD', 'hour': 11, 'flight_ids': ['FL-1176', 'FL-1202', 'FL-1230', 'FL-1231', 'FL-1267', 'FL-1281', 'FL-1295'], 'capacity': 40}]

DEL fog constraints:
[{'airport': 'DEL', 'hour': 6, 'flight_ids': ['FL-1036', 'FL-1041', 'FL-1044', 'FL-1045', 'FL-1046', 'FL-1047', 'FL-1049', 'FL-1050', 'FL-1056', 'FL-1061', 'FL-1062', 'FL-1063', 'FL-1065', 'FL-1066', 'FL-1071', 'FL-1073', 'FL-1075', 'FL-1076', 'FL-1077', 'FL-1078', 'FL-1081', 'FL-1082', 'FL-1083'

In [74]:
airport_capacity_rows = []

for constraint in airport_capacity_constraints:

    flight_ids = constraint["flight_ids"]
    capacity = constraint["capacity"]

    variable_ids = []
    coefficients = []

    for flight_id in flight_ids:

        # Find the cancellation variable
        cancel_id = flight_variables.loc[
            flight_variables["flight_id"] == flight_id,
            "cancel_variable_id"
        ].iloc[0]

        # Movement is counted only if the flight operates:
        # 1 - cancellation
        variable_ids.append(cancel_id)
        coefficients.append(-1)

    airport_capacity_rows.append({
        "airport": constraint["airport"],
        "hour": constraint["hour"],
        "variable_ids": variable_ids,
        "coefficients": coefficients,
        "sense": ">=",
        "rhs": len(flight_ids) - capacity
    })

print("Airport capacity rows:", len(airport_capacity_rows))
print(airport_capacity_rows[:2])

Airport capacity rows: 194
[{'airport': 'AMD', 'hour': 7, 'variable_ids': [np.int64(26508), np.int64(26523), np.int64(26555)], 'coefficients': [-1, -1, -1], 'sense': '>=', 'rhs': -37}, {'airport': 'AMD', 'hour': 8, 'variable_ids': [np.int64(26531), np.int64(26573), np.int64(26574), np.int64(26580), np.int64(26603), np.int64(26612), np.int64(26626), np.int64(26635), np.int64(26640), np.int64(26648), np.int64(26650)], 'coefficients': [-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1], 'sense': '>=', 'rhs': -29}]


In [75]:
del_fog_rows = [
    row for row in airport_capacity_rows
    if row["airport"] == "DEL" and row["rhs"] < 0
]

print("DEL fog rows:", len(del_fog_rows))

for row in del_fog_rows:
    print(row)

DEL fog rows: 19
{'airport': 'DEL', 'hour': 5, 'variable_ids': [np.int64(26488), np.int64(26489), np.int64(26490), np.int64(26491), np.int64(26495), np.int64(26496), np.int64(26498), np.int64(26501), np.int64(26502), np.int64(26504), np.int64(26505), np.int64(26508), np.int64(26515), np.int64(26521), np.int64(26522)], 'coefficients': [-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1], 'sense': '>=', 'rhs': -25}
{'airport': 'DEL', 'hour': 7, 'variable_ids': [np.int64(26492), np.int64(26506), np.int64(26510), np.int64(26600), np.int64(26601), np.int64(26602), np.int64(26603)], 'coefficients': [-1, -1, -1, -1, -1, -1, -1], 'sense': '>=', 'rhs': -3}
{'airport': 'DEL', 'hour': 9, 'variable_ids': [np.int64(26581), np.int64(26589), np.int64(26655), np.int64(26659), np.int64(26691), np.int64(26695), np.int64(26700)], 'coefficients': [-1, -1, -1, -1, -1, -1, -1], 'sense': '>=', 'rhs': -3}
{'airport': 'DEL', 'hour': 10, 'variable_ids': [np.int64(26617), np.int64(26619), np.int64(26621)

In [76]:
del_fog_rows = [
    c for c in airport_capacity_constraints
    if c["airport"] == "DEL" and c["capacity"] == 10
]

print("Actual DEL fog rows:", len(del_fog_rows))

for c in del_fog_rows:
    print(
        "Hour:", c["hour"],
        "| Movements:", len(c["flight_ids"]),
        "| Capacity:", c["capacity"]
    )

Actual DEL fog rows: 4
Hour: 6 | Movements: 34 | Capacity: 10
Hour: 7 | Movements: 7 | Capacity: 10
Hour: 8 | Movements: 15 | Capacity: 10
Hour: 9 | Movements: 7 | Capacity: 10


In [77]:
# PRESOLVER — Rebuild crew lookup structures

# 1. Crew members available for each flight
crew_by_flight = {}

for flight_id, group in crew_variables.groupby("flight_id"):
    crew_by_flight[flight_id] = set(group["crew_id"].astype(str))

# 2. Map (flight, crew) -> global variable ID
crew_variable_lookup = {}

for _, row in crew_variables.iterrows():
    key = (row["flight_id"], str(row["crew_id"]))
    crew_variable_lookup[key] = int(row["variable_id"])

print("Crew lookup structures rebuilt.")
print("Flights in crew_by_flight:", len(crew_by_flight))
print("Crew-variable lookup entries:", len(crew_variable_lookup))

print("\nExpected:")
print("Flights: 789")
print("Lookup entries: 106264")

Crew lookup structures rebuilt.
Flights in crew_by_flight: 789
Crew-variable lookup entries: 106264

Expected:
Flights: 789
Lookup entries: 106264


In [78]:
# PRESOLVER — Rebuild crew conflict constraints

crew_conflict_constraints = []

for _, conflict in crew_conflicts.iterrows():

    flight_1 = conflict["flight_1"]
    flight_2 = conflict["flight_2"]

    crew_1 = crew_by_flight.get(flight_1, set())
    crew_2 = crew_by_flight.get(flight_2, set())

    common_crew = crew_1 & crew_2

    for crew_id in common_crew:

        var_1 = crew_variable_lookup.get((flight_1, crew_id))
        var_2 = crew_variable_lookup.get((flight_2, crew_id))

        if var_1 is not None and var_2 is not None:
            crew_conflict_constraints.append({
                "flight_1": flight_1,
                "flight_2": flight_2,
                "crew_id": crew_id,
                "variable_ids": [var_1, var_2],
                "coefficients": [1, 1],
                "sense": "<=",
                "rhs": 1
            })

print("Crew conflict constraints rebuilt:", len(crew_conflict_constraints))
print("Expected:", 3364254)

Crew conflict constraints rebuilt: 3364254
Expected: 3364254


In [79]:
all_constraint_rows = (
    flight_coverage_constraints
    + aircraft_flow_rows
    + aircraft_outgoing_rows
    + delay_cancel_constraints
    + crew_coverage_constraints
    + crew_conflict_constraints
    + airport_capacity_rows
)

print("Total combined constraints:", len(all_constraint_rows))
print("Expected:", 3443145)

Total combined constraints: 3443145
Expected: 3443145


In [80]:
from scipy.sparse import coo_matrix

print("Variables:", total_variables)
print("Constraints:", len(all_constraint_rows))
print("Sparse matrix shape will be:")
print((len(all_constraint_rows), total_variables))

Variables: 472798
Constraints: 3443145
Sparse matrix shape will be:
(3443145, np.int64(472798))


In [81]:
# PRESOLVER — CELL 40
# Build crew coverage constraint structure

crew_coverage_constraints = []

for _, flight in flight_variables.iterrows():

    flight_id = flight["flight_id"]

    for role in ["CAPTAIN", "FIRST_OFFICER", "CABIN_CREW"]:

        candidate_ids = crew_variables.loc[
            (
                crew_variables["flight_id"] == flight_id
            ) &
            (
                crew_variables["crew_type"] == role
            ),
            "variable_id"
        ].tolist()

        crew_coverage_constraints.append({
            "flight_id": flight_id,
            "crew_role": role,
            "crew_variable_ids": candidate_ids,
            "cancel_variable_id": flight["cancel_variable_id"],
            "rhs": 1
        })

print("Crew coverage constraints:", len(crew_coverage_constraints))
print("Expected:", len(flight_variables) * 3)

print(
    "Constraints with no crew candidates:",
    sum(
        len(c["crew_variable_ids"]) == 0
        for c in crew_coverage_constraints
    )
)

print("\nConstraints by role:")

for role in ["CAPTAIN", "FIRST_OFFICER", "CABIN_CREW"]:
    print(
        role + ":",
        sum(
            c["crew_role"] == role
            for c in crew_coverage_constraints
        )
    )

print("\nExample:")
print(crew_coverage_constraints[0])

Crew coverage constraints: 2367
Expected: 2367
Constraints with no crew candidates: 0

Constraints by role:
CAPTAIN: 789
FIRST_OFFICER: 789
CABIN_CREW: 789

Example:
{'flight_id': 'FL-1000', 'crew_role': 'CAPTAIN', 'crew_variable_ids': [28066, 28067, 28068, 28069, 28070, 28071, 28072, 28073, 28074, 28075, 28076, 28077, 28078, 28079, 28080, 28081, 28082, 28083, 28084, 28085, 28086, 28087, 28088, 28089, 28090, 28091, 28092, 28093, 28094, 28095, 28096, 28097, 28098, 28099, 28100, 28101, 28102, 28103, 28104, 28105, 28106, 28107, 28108, 28109, 28110, 28111, 28112, 28113, 28114, 28115, 28116, 28117, 28118, 28119, 28120, 28121, 28122, 28123, 28124], 'cancel_variable_id': 26488, 'rhs': 1}


In [82]:
# PRESOLVER — CELL 41
# Convert crew coverage constraints to matrix-ready rows

crew_coverage_rows = []

for c in crew_coverage_constraints:

    crew_ids = c["crew_variable_ids"]
    cancel_id = c["cancel_variable_id"]

    crew_coverage_rows.append({
        "flight_id": c["flight_id"],
        "crew_role": c["crew_role"],
        "variable_ids": crew_ids + [cancel_id],
        "coefficients": [1] * len(crew_ids) + [1],
        "sense": "=",
        "rhs": c["rhs"]
    })

print("Crew coverage rows:", len(crew_coverage_rows))
print(crew_coverage_rows[0])

Crew coverage rows: 2367
{'flight_id': 'FL-1000', 'crew_role': 'CAPTAIN', 'variable_ids': [28066, 28067, 28068, 28069, 28070, 28071, 28072, 28073, 28074, 28075, 28076, 28077, 28078, 28079, 28080, 28081, 28082, 28083, 28084, 28085, 28086, 28087, 28088, 28089, 28090, 28091, 28092, 28093, 28094, 28095, 28096, 28097, 28098, 28099, 28100, 28101, 28102, 28103, 28104, 28105, 28106, 28107, 28108, 28109, 28110, 28111, 28112, 28113, 28114, 28115, 28116, 28117, 28118, 28119, 28120, 28121, 28122, 28123, 28124, 26488], 'coefficients': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'sense': '=', 'rhs': 1}


In [83]:
# PRESOLVER — CELL 42
# Convert delay/cancellation constraints to matrix-ready rows

delay_cancel_rows = []

M = 180

for c in delay_cancel_constraints:

    d = c["delay_variable_id"]
    y = c["cancel_variable_id"]

    if "big_m" in c:
        # d + M*y <= M
        delay_cancel_rows.append({
            "flight_id": c["flight_id"],
            "variable_ids": [d, y],
            "coefficients": [1, M],
            "sense": "<=",
            "rhs": M
        })

    else:
        # d + forced_delay*y >= forced_delay
        forced_delay = c["forced_delay"]

        delay_cancel_rows.append({
            "flight_id": c["flight_id"],
            "variable_ids": [d, y],
            "coefficients": [1, forced_delay],
            "sense": ">=",
            "rhs": forced_delay
        })

print("Delay/cancellation rows:", len(delay_cancel_rows))
print(delay_cancel_rows[0])
print(delay_cancel_rows[1])

Delay/cancellation rows: 1578
{'flight_id': 'FL-1000', 'variable_ids': [27277, 26488], 'coefficients': [1, 180], 'sense': '<=', 'rhs': 180}
{'flight_id': 'FL-1000', 'variable_ids': [27277, 26488], 'coefficients': [1, 0], 'sense': '>=', 'rhs': 0}


In [84]:
# PRESOLVER — CELL 43
# Check crew conflict prerequisites

print("crew_conflicts defined:", "crew_conflicts" in globals())
print("crew_by_flight defined:", "crew_by_flight" in globals())
print("crew_variable_lookup defined:", "crew_variable_lookup" in globals())

if "crew_conflicts" in globals():
    print("Crew conflict flight pairs:", len(crew_conflicts))

if "crew_by_flight" in globals():
    print("Flights in crew_by_flight:", len(crew_by_flight))

if "crew_variable_lookup" in globals():
    print("Crew variable lookup entries:", len(crew_variable_lookup))

crew_conflicts defined: True
crew_by_flight defined: True
crew_variable_lookup defined: True
Crew conflict flight pairs: 61120
Flights in crew_by_flight: 789
Crew variable lookup entries: 106264


In [85]:
# PRESOLVER — CELL 44
# Rebuild crew lookup structures for conflict constraints

# Crew IDs that are candidates for each flight
crew_by_flight = (
    crew_candidates
    .groupby("flight_id")["crew_id"]
    .apply(set)
    .to_dict()
)

# Map (flight, crew) -> MILP variable ID
crew_variable_lookup = {
    (row["flight_id"], row["crew_id"]): row["variable_id"]
    for _, row in crew_variables.iterrows()
}

print("Flights in crew_by_flight:", len(crew_by_flight))
print("Crew variable lookup entries:", len(crew_variable_lookup))

print("\nExample crew IDs for FL-1000:")
print(list(crew_by_flight["FL-1000"])[:10])

print("\nExample variable lookup:")
example_crew = next(iter(crew_by_flight["FL-1000"]))
print(
    ("FL-1000", example_crew),
    "->",
    crew_variable_lookup.get(("FL-1000", example_crew))
)

Flights in crew_by_flight: 789
Crew variable lookup entries: 106264

Example crew IDs for FL-1000:
['CRW-0302', 'CRW-0243', 'CRW-0350', 'CRW-0148', 'CRW-0201', 'CRW-0191', 'CRW-0221', 'CRW-0061', 'CRW-0220', 'CRW-0293']

Example variable lookup:
('FL-1000', 'CRW-0302') -> 28212


In [86]:
# PRESOLVER — CELL 45
# Build crew conflict constraints

crew_conflict_constraints = []

for _, conflict in crew_conflicts.iterrows():

    flight_1 = conflict["flight_1"]
    flight_2 = conflict["flight_2"]

    # Crew members who could potentially operate both flights
    common_crew = (
        crew_by_flight[flight_1]
        & crew_by_flight[flight_2]
    )

    for crew_id in common_crew:

        var_1 = crew_variable_lookup[(flight_1, crew_id)]
        var_2 = crew_variable_lookup[(flight_2, crew_id)]

        crew_conflict_constraints.append({
            "flight_1": flight_1,
            "flight_2": flight_2,
            "crew_id": crew_id,
            "variable_ids": [var_1, var_2],
            "coefficients": [1, 1],
            "sense": "<=",
            "rhs": 1
        })

print("Crew conflict constraints:", len(crew_conflict_constraints))
print("Unique crew members involved:",
      len(set(c["crew_id"] for c in crew_conflict_constraints)))

print("\nExample:")
print(crew_conflict_constraints[0])

Crew conflict constraints: 3364254
Unique crew members involved: 388

Example:
{'flight_1': 'FL-1000', 'flight_2': 'FL-1001', 'crew_id': 'CRW-0302', 'variable_ids': [28212, 28405], 'coefficients': [1, 1], 'sense': '<=', 'rhs': 1}


In [87]:
# PRESOLVER — CELL 46
# Convert flight coverage constraints to matrix-ready rows

flight_coverage_rows = []

for c in flight_coverage_constraints:

    variable_ids = (
        c["aircraft_variable_ids"]
        + [c["cancel_variable_id"]]
    )

    flight_coverage_rows.append({
        "flight_id": c["flight_id"],
        "variable_ids": variable_ids,
        "coefficients": [1] * len(variable_ids),
        "sense": "=",
        "rhs": c["rhs"]
    })

print("Flight coverage rows:", len(flight_coverage_rows))
print(flight_coverage_rows[0])

Flight coverage rows: 789
{'flight_id': 'FL-1000', 'variable_ids': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 26488], 'coefficients': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'sense': '=', 'rhs': 1}


In [88]:
# PRESOLVER — CELL 47
# Combine all matrix-ready constraint rows

all_constraint_rows = (
    flight_coverage_rows
    + aircraft_flow_rows
    + aircraft_outgoing_rows
    + delay_cancel_rows
    + crew_coverage_rows
    + crew_conflict_constraints
    + airport_capacity_rows
)

print("Total constraint rows:", len(all_constraint_rows))

# Validate every row has the required matrix fields
incomplete = [
    i for i, row in enumerate(all_constraint_rows)
    if not all(
        key in row
        for key in ["variable_ids", "coefficients", "sense", "rhs"]
    )
]

print("Incomplete rows:", len(incomplete))

if incomplete:
    print("First incomplete row:", incomplete[0])
    print(all_constraint_rows[incomplete[0]])
else:
    print("All constraint rows are matrix-ready.")

Total constraint rows: 3443145
Incomplete rows: 0
All constraint rows are matrix-ready.


In [89]:
# PRESOLVER — CELL 48
# Build sparse constraint matrix using COO format

from scipy.sparse import coo_matrix
import numpy as np

print("Building sparse matrix...")
print("Rows:", len(all_constraint_rows))
print("Columns:", total_variables)

# Count total non-zero entries first
total_nnz = sum(
    len(row["variable_ids"])
    for row in all_constraint_rows
)

print("Expected non-zero entries:", total_nnz)

# Preallocate NumPy arrays
row_indices = np.empty(total_nnz, dtype=np.int32)
col_indices = np.empty(total_nnz, dtype=np.int32)
data = np.empty(total_nnz, dtype=np.float64)

position = 0

for row_idx, row in enumerate(all_constraint_rows):

    n = len(row["variable_ids"])

    row_indices[position:position+n] = row["variable_ids"]
    col_indices[position:position+n] = row["variable_ids"]
    data[position:position+n] = row["coefficients"]

    # Correct row index for every coefficient
    row_indices[position:position+n] = row_idx

    position += n

A = coo_matrix(
    (data, (row_indices, col_indices)),
    shape=(len(all_constraint_rows), total_variables)
).tocsr()

print("\nSparse matrix created.")
print("A shape:", A.shape)
print("Non-zero entries:", A.nnz)
print(
    "Sparsity:",
    A.nnz / (A.shape[0] * A.shape[1])
)

Building sparse matrix...
Rows: 3443145
Columns: 472798
Expected non-zero entries: 7898666

Sparse matrix created.
A shape: (3443145, 472798)
Non-zero entries: 7898666
Sparsity: 4.85202250598651e-06


In [90]:
# PRESOLVER — CELL 49
# Build lower and upper bounds for all constraints

constraint_lb = np.full(
    len(all_constraint_rows),
    -np.inf,
    dtype=np.float64
)

constraint_ub = np.full(
    len(all_constraint_rows),
    np.inf,
    dtype=np.float64
)

for i, row in enumerate(all_constraint_rows):

    if row["sense"] == "=":
        constraint_lb[i] = row["rhs"]
        constraint_ub[i] = row["rhs"]

    elif row["sense"] == "<=":
        constraint_ub[i] = row["rhs"]

    elif row["sense"] == ">=":
        constraint_lb[i] = row["rhs"]

print("Constraint lower bounds:", constraint_lb.shape)
print("Constraint upper bounds:", constraint_ub.shape)

print("\nEquality constraints:",
      np.sum(constraint_lb == constraint_ub))

print("<= constraints:",
      np.sum(np.isneginf(constraint_lb)))

print(">= constraints:",
      np.sum(np.isposinf(constraint_ub)))

print("\nFirst 5 bounds:")
for i in range(5):
    print(
        i,
        "sense =", all_constraint_rows[i]["sense"],
        "lb =", constraint_lb[i],
        "ub =", constraint_ub[i]
    )

Constraint lower bounds: (3443145,)
Constraint upper bounds: (3443145,)

Equality constraints: 3156
<= constraints: 3439006
>= constraints: 983

First 5 bounds:
0 sense = = lb = 1.0 ub = 1.0
1 sense = = lb = 1.0 ub = 1.0
2 sense = = lb = 1.0 ub = 1.0
3 sense = = lb = 1.0 ub = 1.0
4 sense = = lb = 1.0 ub = 1.0


In [91]:
# PRESOLVER — CELL 50
# Validate objective vector

print("Objective vector length:", len(c))
print("Total variables:", total_variables)

print("Non-zero objective coefficients:", np.count_nonzero(c))
print("Zero objective coefficients:", np.sum(c == 0))

print("\nObjective vector valid:", len(c) == total_variables)

Objective vector length: 4
Total variables: 472798
Non-zero objective coefficients: 1
Zero objective coefficients: 0

Objective vector valid: False


In [92]:
# PRESOLVER — CELL 50
# Rebuild objective vector

c = np.zeros(total_variables, dtype=np.float64)

# Cancellation cost
for _, row in flight_variables.iterrows():
    cancel_id = row["cancel_variable_id"]
    c[cancel_id] = row["c_cancel"]

    delay_id = row["delay_variable_id"]
    c[delay_id] = row["c_delay_per_min"]

print("Objective vector length:", len(c))
print("Total variables:", total_variables)
print("Non-zero objective coefficients:", np.count_nonzero(c))
print("Zero objective coefficients:", np.sum(c == 0))

print("\nObjective vector valid:", len(c) == total_variables)

print("\nExample objective coefficients:")
print("Cancellation:", c[flight_variables.iloc[0]["cancel_variable_id"]])
print("Delay:", c[flight_variables.iloc[0]["delay_variable_id"]])

Objective vector length: 472798
Total variables: 472798
Non-zero objective coefficients: 1578
Zero objective coefficients: 471220

Objective vector valid: True

Example objective coefficients:
Cancellation: 690440.0
Delay: 665.0


In [93]:
# PRESOLVER — CELL 51
# Build complete variable bounds and integrality

# Rebuild everything from the actual variable tables
aircraft_variable_ids = aircraft_variables["variable_id"].tolist()
crew_variable_ids = crew_variables["variable_id"].tolist()
aircraft_arc_variable_ids = aircraft_arc_variables["variable_id"].tolist()

# Initialize
lb = np.zeros(total_variables, dtype=np.float64)
ub = np.zeros(total_variables, dtype=np.float64)
integrality = np.zeros(total_variables, dtype=np.int8)

# --------------------------------------------------
# 1. Aircraft assignment variables
# Binary: 0 <= x <= 1
# --------------------------------------------------
for var_id in aircraft_variable_ids:
    lb[var_id] = 0
    ub[var_id] = 1
    integrality[var_id] = 1

# --------------------------------------------------
# 2. Cancellation variables
# Binary: 0 <= y <= 1
# --------------------------------------------------
for _, row in flight_variables.iterrows():
    var_id = int(row["cancel_variable_id"])
    lb[var_id] = 0
    ub[var_id] = 1
    integrality[var_id] = 1

# --------------------------------------------------
# 3. Delay variables
# Continuous: forced_delay <= d <= 180
# --------------------------------------------------
for _, row in flight_variables.iterrows():
    var_id = int(row["delay_variable_id"])
    lb[var_id] = row["forced_delay"]
    ub[var_id] = 180
    integrality[var_id] = 0

# --------------------------------------------------
# 4. Crew assignment variables
# Binary: 0 <= z <= 1
# --------------------------------------------------
for var_id in crew_variable_ids:
    lb[var_id] = 0
    ub[var_id] = 1
    integrality[var_id] = 1

# --------------------------------------------------
# 5. Aircraft sequencing variables
# Binary: 0 <= q <= 1
# --------------------------------------------------
for var_id in aircraft_arc_variable_ids:
    lb[var_id] = 0
    ub[var_id] = 1
    integrality[var_id] = 1

# --------------------------------------------------
# Validation
# --------------------------------------------------

print("Total variables:", total_variables)

print("\nBounds:")
print("  lb length:", len(lb))
print("  ub length:", len(ub))

print("\nIntegrality:")
print("  length:", len(integrality))
print("  integer variables:", np.sum(integrality == 1))
print("  continuous variables:", np.sum(integrality == 0))

print("\nValidation:")
print("  Bounds valid:",
      len(lb) == total_variables and len(ub) == total_variables)

print("  Integrality valid:",
      len(integrality) == total_variables)

print("  lb > ub:", np.sum(lb > ub))

print("\nVariable counts:")
print("  Aircraft assignment:", len(aircraft_variable_ids))
print("  Cancellation:", len(flight_variables))
print("  Delay:", len(flight_variables))
print("  Crew assignment:", len(crew_variable_ids))
print("  Aircraft sequencing:", len(aircraft_arc_variable_ids))

print("\nExpected total:",
      len(aircraft_variable_ids)
      + len(flight_variables)
      + len(flight_variables)
      + len(crew_variable_ids)
      + len(aircraft_arc_variable_ids))

Total variables: 472798

Bounds:
  lb length: 472798
  ub length: 472798

Integrality:
  length: 472798
  integer variables: 472009
  continuous variables: 789

Validation:
  Bounds valid: True
  Integrality valid: True
  lb > ub: 0

Variable counts:
  Aircraft assignment: 26488
  Cancellation: 789
  Delay: 789
  Crew assignment: 106264
  Aircraft sequencing: 338468

Expected total: 472798


In [94]:
# PRESOLVER — CELL 52
# Final MILP consistency check

print("=" * 60)
print("PRESOLVER FINAL CONSISTENCY CHECK")
print("=" * 60)

print("\nVARIABLES")
print("Total variables:", total_variables)
print("c shape:", c.shape)
print("lb shape:", lb.shape)
print("ub shape:", ub.shape)
print("integrality shape:", integrality.shape)

print("\nCONSTRAINTS")
print("A shape:", A.shape)
print("constraint_lb shape:", constraint_lb.shape)
print("constraint_ub shape:", constraint_ub.shape)

print("\nNON-ZERO ENTRIES")
print("A non-zeros:", A.nnz)

print("\nCONSTRAINT TYPES")
print("Equality:", np.sum(constraint_lb == constraint_ub))
print("<= :", np.sum(
    (constraint_ub < np.inf) & (constraint_lb == -np.inf)
))
print(">= :", np.sum(
    (constraint_lb > -np.inf) & (constraint_ub == np.inf)
))
print("Range constraints:", np.sum(
    (constraint_lb > -np.inf) & (constraint_ub < np.inf)
    & (constraint_lb != constraint_ub)
))

print("\nOBJECTIVE")
print("Non-zero coefficients:", np.count_nonzero(c))

print("\nVALIDATION FLAGS")

checks = {
    "A columns == variables":
        A.shape[1] == total_variables,

    "A rows == constraint bounds":
        A.shape[0] == len(constraint_lb) == len(constraint_ub),

    "Objective length correct":
        len(c) == total_variables,

    "Lower bounds length correct":
        len(lb) == total_variables,

    "Upper bounds length correct":
        len(ub) == total_variables,

    "Integrality length correct":
        len(integrality) == total_variables,

    "No lb > ub":
        np.all(lb <= ub),

    "Integrality only 0/1":
        np.all(np.isin(integrality, [0, 1])),

    "No NaN in objective":
        not np.isnan(c).any(),

    "No NaN in bounds":
        not np.isnan(lb).any() and not np.isnan(ub).any(),

    "No NaN in constraint bounds":
        not np.isnan(constraint_lb).any()
        and not np.isnan(constraint_ub).any()
}

for check, result in checks.items():
    print(f"{'PASS' if result else 'FAIL'} — {check}")

print("\nALL CHECKS PASSED:",
      all(checks.values()))

PRESOLVER FINAL CONSISTENCY CHECK

VARIABLES
Total variables: 472798
c shape: (472798,)
lb shape: (472798,)
ub shape: (472798,)
integrality shape: (472798,)

CONSTRAINTS
A shape: (3443145, 472798)
constraint_lb shape: (3443145,)
constraint_ub shape: (3443145,)

NON-ZERO ENTRIES
A non-zeros: 7898666

CONSTRAINT TYPES
Equality: 3156
<= : 3439006
>= : 983
Range constraints: 0

OBJECTIVE
Non-zero coefficients: 1578

VALIDATION FLAGS
PASS — A columns == variables
PASS — A rows == constraint bounds
PASS — Objective length correct
PASS — Lower bounds length correct
PASS — Upper bounds length correct
PASS — Integrality length correct
PASS — No lb > ub
PASS — Integrality only 0/1
PASS — No NaN in objective
PASS — No NaN in bounds
PASS — No NaN in constraint bounds

ALL CHECKS PASSED: True


In [95]:
# PRESOLVER — CELL 53
# Audit flight coverage constraints

print("=" * 60)
print("FLIGHT COVERAGE AUDIT")
print("=" * 60)

errors = []

for row in flight_coverage_rows:

    flight_id = row["flight_id"]
    variable_ids = row["variable_ids"]
    coefficients = row["coefficients"]

    # Expected: aircraft variables + exactly one cancellation variable
    aircraft_ids = set(
        aircraft_variables.loc[
            aircraft_variables["flight_id"] == flight_id,
            "variable_id"
        ].tolist()
    )

    cancel_id = int(
        flight_variables.loc[
            flight_variables["flight_id"] == flight_id,
            "cancel_variable_id"
        ].iloc[0]
    )

    expected_ids = aircraft_ids | {cancel_id}

    # Check variable set
    if set(variable_ids) != expected_ids:
        errors.append(
            f"{flight_id}: variable mismatch"
        )

    # Check all coefficients are +1
    if any(coef != 1 for coef in coefficients):
        errors.append(
            f"{flight_id}: coefficient not +1"
        )

    # Check equality and RHS
    if row["sense"] != "=" or row["rhs"] != 1:
        errors.append(
            f"{flight_id}: incorrect sense/RHS"
        )

print("Total coverage constraints:", len(flight_coverage_rows))
print("Expected:", len(flights))

print("Flights checked:", len(set(
    row["flight_id"] for row in flight_coverage_rows
)))

print("Errors:", len(errors))

if errors:
    print("\nFirst errors:")
    for error in errors[:10]:
        print(error)
else:
    print("\nALL FLIGHT COVERAGE CONSTRAINTS PASSED")

FLIGHT COVERAGE AUDIT


Total coverage constraints: 789
Expected: 789
Flights checked: 789
Errors: 0

ALL FLIGHT COVERAGE CONSTRAINTS PASSED


In [96]:
# PRESOLVER — CELL 54
# Audit aircraft flow constraints

print("=" * 60)
print("AIRCRAFT FLOW AUDIT")
print("=" * 60)

errors = []

# --------------------------------------------------
# Audit incoming-flow constraints
# --------------------------------------------------

for row in aircraft_flow_rows:

    variable_ids = row["variable_ids"]
    coefficients = row["coefficients"]
    sense = row["sense"]
    rhs = row["rhs"]

    x_id = variable_ids[0]
    incoming_ids = variable_ids[1:]

    # x coefficient must be -1
    if coefficients[0] != -1:
        errors.append(
            f"Incoming row for x={x_id}: x coefficient is not -1"
        )

    # Every incoming arc coefficient must be +1
    if any(c != 1 for c in coefficients[1:]):
        errors.append(
            f"Incoming row for x={x_id}: invalid incoming coefficient"
        )

    # Must be <= 0
    if sense != "<=" or rhs != 0:
        errors.append(
            f"Incoming row for x={x_id}: incorrect sense/RHS"
        )

# --------------------------------------------------
# Audit mandatory-predecessor constraints
# --------------------------------------------------

mandatory_rows = 0

for row in aircraft_flow_rows:

    variable_ids = row["variable_ids"]
    coefficients = row["coefficients"]

    # Form: x - incoming <= 0
    if coefficients[0] == 1:

        mandatory_rows += 1

        if any(c != -1 for c in coefficients[1:]):
            errors.append(
                f"Mandatory predecessor row for x={variable_ids[0]}: "
                f"invalid incoming coefficient"
            )

        if row["sense"] != "<=" or row["rhs"] != 0:
            errors.append(
                f"Mandatory predecessor row for x={variable_ids[0]}: "
                f"incorrect sense/RHS"
            )

# --------------------------------------------------
# Audit outgoing-flow constraints
# --------------------------------------------------

for row in aircraft_outgoing_rows:

    variable_ids = row["variable_ids"]
    coefficients = row["coefficients"]

    # Last variable is x
    x_id = variable_ids[-1]
    outgoing_ids = variable_ids[:-1]

    # Every outgoing arc coefficient must be +1
    if any(c != 1 for c in coefficients[:-1]):
        errors.append(
            f"Outgoing row for x={x_id}: invalid outgoing coefficient"
        )

    # x coefficient must be -1
    if coefficients[-1] != -1:
        errors.append(
            f"Outgoing row for x={x_id}: x coefficient is not -1"
        )

    # Must be <= 0
    if row["sense"] != "<=" or row["rhs"] != 0:
        errors.append(
            f"Outgoing row for x={x_id}: incorrect sense/RHS"
        )

print("Incoming/mandatory flow rows:", len(aircraft_flow_rows))
print("Outgoing flow rows:", len(aircraft_outgoing_rows))
print("Mandatory predecessor rows:", mandatory_rows)

print("Errors:", len(errors))

if errors:
    print("\nFirst errors:")
    for error in errors[:10]:
        print(error)
else:
    print("\nALL AIRCRAFT FLOW CONSTRAINTS PASSED")

AIRCRAFT FLOW AUDIT
Incoming/mandatory flow rows: 47475
Outgoing flow rows: 26488
Mandatory predecessor rows: 45305
Errors: 112784

First errors:
Incoming row for x=144896: x coefficient is not -1
Incoming row for x=144896: invalid incoming coefficient
Incoming row for x=1560: x coefficient is not -1
Incoming row for x=1560: invalid incoming coefficient
Incoming row for x=144897: x coefficient is not -1
Incoming row for x=144897: invalid incoming coefficient
Incoming row for x=1561: x coefficient is not -1
Incoming row for x=1561: invalid incoming coefficient
Incoming row for x=144898: x coefficient is not -1
Incoming row for x=144898: invalid incoming coefficient


In [97]:
# PRESOLVER — CELL 54A
# Inspect actual aircraft-flow constraint structure

print("=" * 60)
print("AIRCRAFT FLOW ROW STRUCTURE")
print("=" * 60)

for i, row in enumerate(aircraft_flow_rows[:10]):

    print(f"\nROW {i}")
    print("variable_ids :", row["variable_ids"][:10],
          "..." if len(row["variable_ids"]) > 10 else "")
    print("coefficients :", row["coefficients"][:10],
          "..." if len(row["coefficients"]) > 10 else "")
    print("sense        :", row["sense"])
    print("rhs          :", row["rhs"])

print("\n--- LAST 5 ROWS ---")

for i, row in enumerate(aircraft_flow_rows[-5:], start=len(aircraft_flow_rows)-5):

    print(f"\nROW {i}")
    print("variable_ids :", row["variable_ids"][:10],
          "..." if len(row["variable_ids"]) > 10 else "")
    print("coefficients :", row["coefficients"][:10],
          "..." if len(row["coefficients"]) > 10 else "")
    print("sense        :", row["sense"])
    print("rhs          :", row["rhs"])

AIRCRAFT FLOW ROW STRUCTURE

ROW 0
variable_ids : [0] 
coefficients : [-1] 
sense        : <=
rhs          : 0

ROW 1
variable_ids : [1] 
coefficients : [-1] 
sense        : <=
rhs          : 0

ROW 2
variable_ids : [2] 
coefficients : [-1] 
sense        : <=
rhs          : 0

ROW 3
variable_ids : [3] 
coefficients : [-1] 
sense        : <=
rhs          : 0

ROW 4
variable_ids : [4] 
coefficients : [-1] 
sense        : <=
rhs          : 0

ROW 5
variable_ids : [5] 
coefficients : [-1] 
sense        : <=
rhs          : 0

ROW 6
variable_ids : [6] 
coefficients : [-1] 
sense        : <=
rhs          : 0

ROW 7
variable_ids : [7] 
coefficients : [-1] 
sense        : <=
rhs          : 0

ROW 8
variable_ids : [8] 
coefficients : [-1] 
sense        : <=
rhs          : 0

ROW 9
variable_ids : [9] 
coefficients : [-1] 
sense        : <=
rhs          : 0

--- LAST 5 ROWS ---

ROW 47470
variable_ids : [183629, 185226, 202093, 225376, 269031, 271645, 277164, 313101, 322265, 331600] ...
coefficien

In [98]:
# PRESOLVER — CELL 54B
# Correct audit of aircraft flow constraints

print("=" * 60)
print("CORRECT AIRCRAFT FLOW AUDIT")
print("=" * 60)

errors = []

incoming_rows = 0
mandatory_predecessor_rows = 0
start_rows = 0

# --------------------------------------------------
# Audit all aircraft flow rows
# --------------------------------------------------

for row in aircraft_flow_rows:

    vars_ = row["variable_ids"]
    coeffs = row["coefficients"]

    if row["sense"] != "<=" or row["rhs"] != 0:
        errors.append("Incorrect sense/RHS")
        continue

    # Case 1:
    # -x <= 0
    # Startable flight with no incoming arc
    if len(coeffs) == 1 and coeffs[0] == -1:
        start_rows += 1
        continue

    # Case 2:
    # sum(incoming) - x <= 0
    # All coefficients +1 except final -1
    if coeffs[-1] == -1 and all(c == 1 for c in coeffs[:-1]):
        incoming_rows += 1
        continue

    # Case 3:
    # x - sum(incoming) <= 0
    # First coefficient +1, remaining -1
    if coeffs[0] == 1 and all(c == -1 for c in coeffs[1:]):
        mandatory_predecessor_rows += 1
        continue

    errors.append(
        f"Unrecognized row structure: "
        f"vars={len(vars_)}, coeffs={coeffs[:10]}"
    )

# --------------------------------------------------
# Audit outgoing rows
# --------------------------------------------------

outgoing_errors = []

for row in aircraft_outgoing_rows:

    coeffs = row["coefficients"]

    if row["sense"] != "<=" or row["rhs"] != 0:
        outgoing_errors.append("Incorrect sense/RHS")
        continue

    # sum(outgoing) - x <= 0
    if len(coeffs) < 1:
        outgoing_errors.append("Empty outgoing row")
        continue

    if coeffs[-1] != -1:
        outgoing_errors.append(
            f"x coefficient is not -1: {coeffs}"
        )

    if any(c != 1 for c in coeffs[:-1]):
        outgoing_errors.append(
            f"Invalid outgoing coefficient: {coeffs}"
        )

print("\nINCOMING FLOW")
print("Total rows:", len(aircraft_flow_rows))
print("Start rows:", start_rows)
print("Incoming capacity rows:", incoming_rows)
print("Mandatory predecessor rows:", mandatory_predecessor_rows)

print("\nOUTGOING FLOW")
print("Total rows:", len(aircraft_outgoing_rows))
print("Outgoing errors:", len(outgoing_errors))

print("\nTOTAL ERRORS:", len(errors) + len(outgoing_errors))

if errors:
    print("\nFirst incoming-flow errors:")
    for e in errors[:10]:
        print(e)

if outgoing_errors:
    print("\nFirst outgoing-flow errors:")
    for e in outgoing_errors[:10]:
        print(e)

if not errors and not outgoing_errors:
    print("\nALL AIRCRAFT FLOW CONSTRAINTS PASSED")

CORRECT AIRCRAFT FLOW AUDIT

INCOMING FLOW
Total rows: 47475
Start rows: 2170
Incoming capacity rows: 26288
Mandatory predecessor rows: 19017

OUTGOING FLOW
Total rows: 26488
Outgoing errors: 0

TOTAL ERRORS: 0

ALL AIRCRAFT FLOW CONSTRAINTS PASSED


In [99]:
# PRESOLVER — CELL 55
# Audit delay/cancellation constraints

print("=" * 60)
print("DELAY / CANCELLATION AUDIT")
print("=" * 60)

errors = []

# Every flight should have exactly 2 constraints
if len(delay_cancel_rows) != 2 * len(flight_variables):
    errors.append(
        f"Expected {2 * len(flight_variables)} rows, "
        f"got {len(delay_cancel_rows)}"
    )

for flight_id in flight_variables["flight_id"]:

    flight_rows = [
        row for row in delay_cancel_rows
        if row["flight_id"] == flight_id
    ]

    if len(flight_rows) != 2:
        errors.append(
            f"{flight_id}: expected 2 rows, got {len(flight_rows)}"
        )
        continue

    flight = flight_variables[
        flight_variables["flight_id"] == flight_id
    ].iloc[0]

    delay_id = int(flight["delay_variable_id"])
    cancel_id = int(flight["cancel_variable_id"])
    forced = int(flight["forced_delay"])

    # Identify <= row
    upper_rows = [
        r for r in flight_rows
        if r["sense"] == "<="
    ]

    # Identify >= row
    lower_rows = [
        r for r in flight_rows
        if r["sense"] == ">="
    ]

    if len(upper_rows) != 1:
        errors.append(f"{flight_id}: incorrect number of <= rows")
    else:
        row = upper_rows[0]

        expected_ids = [delay_id, cancel_id]
        expected_coeffs = [1, 180]

        if row["variable_ids"] != expected_ids:
            errors.append(
                f"{flight_id}: incorrect <= variable IDs"
            )

        if row["coefficients"] != expected_coeffs:
            errors.append(
                f"{flight_id}: incorrect <= coefficients"
            )

        if row["rhs"] != 180:
            errors.append(
                f"{flight_id}: incorrect <= RHS"
            )

    if len(lower_rows) != 1:
        errors.append(f"{flight_id}: incorrect number of >= rows")
    else:
        row = lower_rows[0]

        expected_ids = [delay_id, cancel_id]
        expected_coeffs = [1, forced]

        if row["variable_ids"] != expected_ids:
            errors.append(
                f"{flight_id}: incorrect >= variable IDs"
            )

        if row["coefficients"] != expected_coeffs:
            errors.append(
                f"{flight_id}: incorrect >= coefficients"
            )

        if row["rhs"] != forced:
            errors.append(
                f"{flight_id}: incorrect >= RHS"
            )

print("Flights checked:", len(flight_variables))
print("Constraint rows:", len(delay_cancel_rows))
print("Expected rows:", 2 * len(flight_variables))
print("Errors:", len(errors))

if errors:
    print("\nFirst errors:")
    for error in errors[:10]:
        print(error)
else:
    print("\nALL DELAY/CANCELLATION CONSTRAINTS PASSED")

DELAY / CANCELLATION AUDIT
Flights checked: 789
Constraint rows: 1578
Expected rows: 1578
Errors: 0

ALL DELAY/CANCELLATION CONSTRAINTS PASSED


In [100]:
# PRESOLVER — CELL 56
# Build crew coverage constraints

print("=" * 60)
print("BUILDING CREW COVERAGE CONSTRAINTS")
print("=" * 60)

crew_coverage_constraints = []

for _, flight in flight_variables.iterrows():

    flight_id = flight["flight_id"]
    cancel_id = int(flight["cancel_variable_id"])

    for role in ["CAPTAIN", "FIRST_OFFICER", "CABIN_CREW"]:

        candidate_ids = crew_variables.loc[
            (crew_variables["flight_id"] == flight_id) &
            (crew_variables["crew_type"] == role),
            "variable_id"
        ].astype(int).tolist()

        variable_ids = candidate_ids + [cancel_id]
        coefficients = [1] * len(variable_ids)

        crew_coverage_constraints.append({
            "flight_id": flight_id,
            "crew_role": role,
            "variable_ids": variable_ids,
            "coefficients": coefficients,
            "sense": "=",
            "rhs": 1
        })

print("Flights:", len(flight_variables))
print("Crew roles:", 3)
print("Crew coverage constraints:", len(crew_coverage_constraints))
print("Expected:", len(flight_variables) * 3)

print("\nExample:")
print(crew_coverage_constraints[0])

print("\nCREW COVERAGE CONSTRAINTS BUILT")

BUILDING CREW COVERAGE CONSTRAINTS
Flights: 789
Crew roles: 3
Crew coverage constraints: 2367
Expected: 2367

Example:
{'flight_id': 'FL-1000', 'crew_role': 'CAPTAIN', 'variable_ids': [28066, 28067, 28068, 28069, 28070, 28071, 28072, 28073, 28074, 28075, 28076, 28077, 28078, 28079, 28080, 28081, 28082, 28083, 28084, 28085, 28086, 28087, 28088, 28089, 28090, 28091, 28092, 28093, 28094, 28095, 28096, 28097, 28098, 28099, 28100, 28101, 28102, 28103, 28104, 28105, 28106, 28107, 28108, 28109, 28110, 28111, 28112, 28113, 28114, 28115, 28116, 28117, 28118, 28119, 28120, 28121, 28122, 28123, 28124, 26488], 'coefficients': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'sense': '=', 'rhs': 1}

CREW COVERAGE CONSTRAINTS BUILT


In [101]:
# PRESOLVER — CELL 57
# Convert crew coverage constraints into matrix-ready rows

crew_coverage_rows = []

for constraint in crew_coverage_constraints:
    crew_coverage_rows.append({
        "flight_id": constraint["flight_id"],
        "crew_role": constraint["crew_role"],
        "variable_ids": constraint["variable_ids"],
        "coefficients": constraint["coefficients"],
        "sense": constraint["sense"],
        "rhs": constraint["rhs"]
    })

print("=" * 60)
print("CREW COVERAGE ROWS")
print("=" * 60)

print("Rows:", len(crew_coverage_rows))
print("Expected:", len(flight_variables) * 3)

print("\nExample:")
print(crew_coverage_rows[0])

print("\nCREW COVERAGE ROWS READY")

CREW COVERAGE ROWS
Rows: 2367
Expected: 2367

Example:
{'flight_id': 'FL-1000', 'crew_role': 'CAPTAIN', 'variable_ids': [28066, 28067, 28068, 28069, 28070, 28071, 28072, 28073, 28074, 28075, 28076, 28077, 28078, 28079, 28080, 28081, 28082, 28083, 28084, 28085, 28086, 28087, 28088, 28089, 28090, 28091, 28092, 28093, 28094, 28095, 28096, 28097, 28098, 28099, 28100, 28101, 28102, 28103, 28104, 28105, 28106, 28107, 28108, 28109, 28110, 28111, 28112, 28113, 28114, 28115, 28116, 28117, 28118, 28119, 28120, 28121, 28122, 28123, 28124, 26488], 'coefficients': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'sense': '=', 'rhs': 1}

CREW COVERAGE ROWS READY


In [102]:
# PRESOLVER — CELL 58
# Build crew conflict constraints for overlapping flights

print("=" * 60)
print("BUILDING CREW CONFLICT CONSTRAINTS")
print("=" * 60)

crew_conflict_constraints = []

for _, conflict in crew_conflicts.iterrows():

    flight_1 = conflict["flight_1"]
    flight_2 = conflict["flight_2"]

    # Crew members who are candidates for both flights
    common_crew = set(
        crew_by_flight.get(flight_1, [])
    ).intersection(
        crew_by_flight.get(flight_2, [])
    )

    for crew_id in common_crew:

        var_1 = crew_variable_lookup.get(
            (flight_1, crew_id)
        )

        var_2 = crew_variable_lookup.get(
            (flight_2, crew_id)
        )

        if var_1 is None or var_2 is None:
            continue

        crew_conflict_constraints.append({
            "flight_1": flight_1,
            "flight_2": flight_2,
            "crew_id": crew_id,
            "variable_ids": [int(var_1), int(var_2)],
            "coefficients": [1, 1],
            "sense": "<=",
            "rhs": 1
        })

print("Flight overlap pairs:", len(crew_conflicts))
print("Crew conflict constraints:", len(crew_conflict_constraints))
print(
    "Unique crew members involved:",
    len(set(r["crew_id"] for r in crew_conflict_constraints))
)

print("\nExample:")
print(crew_conflict_constraints[0])

print("\nCREW CONFLICT CONSTRAINTS BUILT")

BUILDING CREW CONFLICT CONSTRAINTS
Flight overlap pairs: 61120
Crew conflict constraints: 3364254
Unique crew members involved: 388

Example:
{'flight_1': 'FL-1000', 'flight_2': 'FL-1001', 'crew_id': 'CRW-0302', 'variable_ids': [28212, 28405], 'coefficients': [1, 1], 'sense': '<=', 'rhs': 1}

CREW CONFLICT CONSTRAINTS BUILT


In [103]:
# PRESOLVER — CELL 59
# Convert crew conflict constraints into matrix-ready rows

crew_conflict_rows = []

for constraint in crew_conflict_constraints:
    crew_conflict_rows.append({
        "flight_1": constraint["flight_1"],
        "flight_2": constraint["flight_2"],
        "crew_id": constraint["crew_id"],
        "variable_ids": constraint["variable_ids"],
        "coefficients": constraint["coefficients"],
        "sense": constraint["sense"],
        "rhs": constraint["rhs"]
    })

print("=" * 60)
print("CREW CONFLICT ROWS")
print("=" * 60)

print("Rows:", len(crew_conflict_rows))
print("Expected:", len(crew_conflict_constraints))

print("\nExample:")
print(crew_conflict_rows[0])

print("\nCREW CONFLICT ROWS READY")

CREW CONFLICT ROWS
Rows: 3364254
Expected: 3364254

Example:
{'flight_1': 'FL-1000', 'flight_2': 'FL-1001', 'crew_id': 'CRW-0302', 'variable_ids': [28212, 28405], 'coefficients': [1, 1], 'sense': '<=', 'rhs': 1}

CREW CONFLICT ROWS READY


In [104]:
# PRESOLVER — CELL 60
# Build airport movement candidates

print("=" * 60)
print("BUILDING AIRPORT MOVEMENT CANDIDATES")
print("=" * 60)

# Get cancellation variable IDs from flight_variables
flight_cancel_info = flight_variables[
    ["flight_id", "cancel_variable_id"]
].copy()

departure_movements = flights[
    ["flight_id", "origin", "std"]
].merge(
    flight_cancel_info,
    on="flight_id",
    how="left"
)

departure_movements["movement_type"] = "DEPARTURE"
departure_movements["airport"] = departure_movements["origin"]
departure_movements["time_min"] = departure_movements["std"]

arrival_movements = flights[
    ["flight_id", "destination", "sta"]
].merge(
    flight_cancel_info,
    on="flight_id",
    how="left"
)

arrival_movements["movement_type"] = "ARRIVAL"
arrival_movements["airport"] = arrival_movements["destination"]
arrival_movements["time_min"] = arrival_movements["sta"]

airport_movements = pd.concat(
    [departure_movements, arrival_movements],
    ignore_index=True
)

airport_movements["hour"] = (
    airport_movements["time_min"] // 60
).astype(int)

airport_movements = airport_movements[
    [
        "flight_id",
        "airport",
        "movement_type",
        "time_min",
        "hour",
        "cancel_variable_id"
    ]
].copy()

print("Total movements:", len(airport_movements))
print("Expected:", 2 * len(flights))
print("Unique airports:", airport_movements["airport"].nunique())
print("Hourly bins:", airport_movements["hour"].nunique())

print("\nMovement types:")
print(airport_movements["movement_type"].value_counts())

print("\nExample:")
print(airport_movements.head())

print("\nAIRPORT MOVEMENT CANDIDATES BUILT")

BUILDING AIRPORT MOVEMENT CANDIDATES
Total movements: 1578
Expected: 1578
Unique airports: 10
Hourly bins: 21

Movement types:
movement_type
DEPARTURE    789
ARRIVAL      789
Name: count, dtype: int64

Example:
  flight_id airport movement_type  time_min  hour  cancel_variable_id
0   FL-1000     DEL     DEPARTURE       330     5               26488
1   FL-1001     DEL     DEPARTURE       331     5               26489
2   FL-1002     DEL     DEPARTURE       331     5               26490
3   FL-1003     DEL     DEPARTURE       333     5               26491
4   FL-1004     HYD     DEPARTURE       333     5               26492

AIRPORT MOVEMENT CANDIDATES BUILT


In [105]:
# PRESOLVER — CELL 61
# Build airport capacity constraints

print("=" * 60)
print("BUILDING AIRPORT CAPACITY CONSTRAINTS")
print("=" * 60)

# Reload airport capacity data with an unambiguous variable name
airport_capacity = pd.read_csv("airport_capacity.csv")

capacity_lookup = (
    airport_capacity
    .set_index("airport")
    .to_dict("index")
)

airport_capacity_rows = []

for (airport_name, hour), group in airport_movements.groupby(
    ["airport", "hour"]
):

    capacity_info = capacity_lookup[airport_name]

    normal_capacity = int(
        capacity_info["normal_hourly_capacity"]
    )

    fog_capacity = int(
        capacity_info["fog_window_hourly_capacity"]
    )

    disruption_start = capacity_info["disruption_start_min"]
    disruption_end = capacity_info["disruption_end_min"]

    hour_start = hour * 60
    hour_end = hour_start + 59

    fog_active = (
        pd.notna(disruption_start)
        and pd.notna(disruption_end)
        and hour_start <= disruption_end
        and hour_end >= disruption_start
    )

    capacity = fog_capacity if fog_active else normal_capacity

    flight_ids = group["flight_id"].tolist()
    cancel_ids = (
        group["cancel_variable_id"]
        .astype(int)
        .tolist()
    )

    # sum(1 - y_f) <= capacity
    #
    # -sum(y_f) >= capacity - number_of_movements
    rhs = capacity - len(flight_ids)

    airport_capacity_rows.append({
        "airport": airport_name,
        "hour": int(hour),
        "movement_count": len(flight_ids),
        "capacity": capacity,
        "variable_ids": cancel_ids,
        "coefficients": [-1] * len(cancel_ids),
        "sense": ">=",
        "rhs": rhs
    })

print("Airport capacity rows:", len(airport_capacity_rows))

print("\nRows by capacity type:")
print(
    pd.Series(
        [
            "FOG" if (
                r["capacity"] ==
                int(capacity_lookup[r["airport"]]["fog_window_hourly_capacity"])
                and pd.notna(
                    capacity_lookup[r["airport"]]["disruption_start_min"]
                )
            ) else "NORMAL"
            for r in airport_capacity_rows
        ]
    ).value_counts()
)

print("\nOver-capacity bins:",
      sum(
          r["movement_count"] > r["capacity"]
          for r in airport_capacity_rows
      )
)

print("\nExample over-capacity row:")
for row in airport_capacity_rows:
    if row["movement_count"] > row["capacity"]:
        print(row)
        break

print("\nAIRPORT CAPACITY CONSTRAINTS BUILT")

BUILDING AIRPORT CAPACITY CONSTRAINTS
Airport capacity rows: 194

Rows by capacity type:
NORMAL    190
FOG         4
Name: count, dtype: int64

Over-capacity bins: 2

Example over-capacity row:
{'airport': 'DEL', 'hour': 6, 'movement_count': 34, 'capacity': 10, 'variable_ids': [26524, 26529, 26532, 26533, 26534, 26535, 26537, 26538, 26544, 26549, 26550, 26551, 26553, 26554, 26559, 26561, 26563, 26564, 26565, 26566, 26569, 26570, 26571, 26572, 26573, 26574, 26575, 26590, 26591, 26592, 26594, 26597, 26598, 26599], 'coefficients': [-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1], 'sense': '>=', 'rhs': -24}

AIRPORT CAPACITY CONSTRAINTS BUILT


In [106]:
# PRESOLVER — CELL 62
# Combine all constraint families

print("=" * 60)
print("ASSEMBLING COMPLETE MILP CONSTRAINT SET")
print("=" * 60)

all_constraint_rows = (
    flight_coverage_rows
    + aircraft_flow_rows
    + aircraft_outgoing_rows
    + delay_cancel_rows
    + crew_coverage_rows
    + crew_conflict_rows
    + airport_capacity_rows
)

print("Total constraint rows:", len(all_constraint_rows))

print("\nConstraint breakdown:")
print("Flight coverage:", len(flight_coverage_rows))
print("Aircraft flow:", len(aircraft_flow_rows))
print("Aircraft outgoing:", len(aircraft_outgoing_rows))
print("Delay/cancellation:", len(delay_cancel_rows))
print("Crew coverage:", len(crew_coverage_rows))
print("Crew conflicts:", len(crew_conflict_rows))
print("Airport capacity:", len(airport_capacity_rows))

print("\nExample final row:")
print(all_constraint_rows[-1])

print("\nALL CONSTRAINT FAMILIES COMBINED")

ASSEMBLING COMPLETE MILP CONSTRAINT SET
Total constraint rows: 3443145

Constraint breakdown:
Flight coverage: 789
Aircraft flow: 47475
Aircraft outgoing: 26488
Delay/cancellation: 1578
Crew coverage: 2367
Crew conflicts: 3364254
Airport capacity: 194

Example final row:
{'airport': 'PNQ', 'hour': 25, 'movement_count': 1, 'capacity': 40, 'variable_ids': [27270], 'coefficients': [-1], 'sense': '>=', 'rhs': 39}

ALL CONSTRAINT FAMILIES COMBINED


In [107]:
# PRESOLVER — CELL 63
# Build sparse constraint matrix A

print("=" * 60)
print("BUILDING SPARSE CONSTRAINT MATRIX")
print("=" * 60)

from scipy.sparse import coo_matrix

row_indices = []
col_indices = []
data = []

for row_idx, row in enumerate(all_constraint_rows):

    variable_ids = row["variable_ids"]
    coefficients = row["coefficients"]

    for variable_id, coefficient in zip(
        variable_ids,
        coefficients
    ):
        row_indices.append(row_idx)
        col_indices.append(int(variable_id))
        data.append(float(coefficient))

A = coo_matrix(
    (
        data,
        (row_indices, col_indices)
    ),
    shape=(
        len(all_constraint_rows),
        total_variables
    )
).tocsr()

print("A shape:", A.shape)
print("Non-zero entries:", A.nnz)

total_elements = A.shape[0] * A.shape[1]
sparsity = 1 - (A.nnz / total_elements)

print("Sparsity:", sparsity)
print("Density:", A.nnz / total_elements)

print("\nSparse matrix format:", type(A).__name__)
print("\nSPARSE MATRIX A BUILT")

BUILDING SPARSE CONSTRAINT MATRIX
A shape: (3443145, 472798)
Non-zero entries: 7898666
Sparsity: 0.999995147977494
Density: 4.85202250598651e-06

Sparse matrix format: csr_matrix

SPARSE MATRIX A BUILT


In [108]:
# PRESOLVER — CELL 64
# Build lower and upper bounds for constraints

print("=" * 60)
print("BUILDING CONSTRAINT BOUNDS")
print("=" * 60)

constraint_lower = []
constraint_upper = []

for row in all_constraint_rows:

    rhs = float(row["rhs"])
    sense = row["sense"]

    if sense == "=":
        constraint_lower.append(rhs)
        constraint_upper.append(rhs)

    elif sense == "<=":
        constraint_lower.append(-float("inf"))
        constraint_upper.append(rhs)

    elif sense == ">=":
        constraint_lower.append(rhs)
        constraint_upper.append(float("inf"))

    else:
        raise ValueError(
            f"Unknown constraint sense: {sense}"
        )

constraint_lower = np.array(
    constraint_lower,
    dtype=float
)

constraint_upper = np.array(
    constraint_upper,
    dtype=float
)

print("Lower-bound shape:", constraint_lower.shape)
print("Upper-bound shape:", constraint_upper.shape)

print("\nEquality constraints:",
      np.sum(constraint_lower == constraint_upper))

print("<= constraints:",
      np.sum(np.isneginf(constraint_lower) &
             np.isfinite(constraint_upper)))

print(">= constraints:",
      np.sum(np.isfinite(constraint_lower) &
             np.isposinf(constraint_upper)))

print("\nTotal constraints:", len(constraint_lower))
print("Expected:", A.shape[0])

print("\nCONSTRAINT BOUNDS BUILT")

BUILDING CONSTRAINT BOUNDS
Lower-bound shape: (3443145,)
Upper-bound shape: (3443145,)

Equality constraints: 3156
<= constraints: 3439006
>= constraints: 983

Total constraints: 3443145
Expected: 3443145

CONSTRAINT BOUNDS BUILT


In [109]:
# PRESOLVER — CELL 65
# Build variable bounds and integrality

print("=" * 60)
print("BUILDING VARIABLE BOUNDS & INTEGRALITY")
print("=" * 60)

# Initialize
variable_lower = np.zeros(total_variables, dtype=float)
variable_upper = np.zeros(total_variables, dtype=float)
variable_integrality = np.zeros(total_variables, dtype=int)

# --------------------------------------------------
# Aircraft assignment variables: binary
# --------------------------------------------------

for _, row in aircraft_variables.iterrows():
    idx = int(row["variable_id"])
    variable_lower[idx] = 0
    variable_upper[idx] = 1
    variable_integrality[idx] = 1

# --------------------------------------------------
# Cancellation variables: binary
# --------------------------------------------------

for _, row in flight_variables.iterrows():
    idx = int(row["cancel_variable_id"])
    variable_lower[idx] = 0
    variable_upper[idx] = 1
    variable_integrality[idx] = 1

# --------------------------------------------------
# Delay variables: continuous
# --------------------------------------------------

for _, row in flight_variables.iterrows():
    idx = int(row["delay_variable_id"])
    variable_lower[idx] = float(row["delay_lb"])
    variable_upper[idx] = float(row["delay_ub"])
    variable_integrality[idx] = 0

# --------------------------------------------------
# Crew assignment variables: binary
# --------------------------------------------------

for _, row in crew_variables.iterrows():
    idx = int(row["variable_id"])
    variable_lower[idx] = 0
    variable_upper[idx] = 1
    variable_integrality[idx] = 1

# --------------------------------------------------
# Aircraft sequencing variables: binary
# --------------------------------------------------

for _, row in aircraft_arc_variables.iterrows():
    idx = int(row["variable_id"])
    variable_lower[idx] = 0
    variable_upper[idx] = 1
    variable_integrality[idx] = 1

print("Total variables:", total_variables)
print("Lower-bound length:", len(variable_lower))
print("Upper-bound length:", len(variable_upper))
print("Integrality length:", len(variable_integrality))

print("\nInteger variables:",
      np.sum(variable_integrality == 1))

print("Continuous variables:",
      np.sum(variable_integrality == 0))

print("\nVariables with bounds [0,1]:",
      np.sum(
          (variable_lower == 0) &
          (variable_upper == 1)
      ))

print("\nAny lower bound > upper bound:",
      np.any(variable_lower > variable_upper))

print("\nVARIABLE BOUNDS & INTEGRALITY BUILT")

BUILDING VARIABLE BOUNDS & INTEGRALITY
Total variables: 472798
Lower-bound length: 472798
Upper-bound length: 472798
Integrality length: 472798

Integer variables: 472009
Continuous variables: 789

Variables with bounds [0,1]: 472009

Any lower bound > upper bound: False

VARIABLE BOUNDS & INTEGRALITY BUILT


In [110]:
# PRESOLVER — CELL 66
# Build objective vector c

print("=" * 60)
print("BUILDING OBJECTIVE VECTOR")
print("=" * 60)

objective = np.zeros(total_variables, dtype=float)

# Cancellation and delay costs
for _, row in flight_variables.iterrows():

    cancel_id = int(row["cancel_variable_id"])
    delay_id = int(row["delay_variable_id"])

    objective[cancel_id] = float(row["c_cancel"])
    objective[delay_id] = float(row["c_delay_per_min"])

print("Objective length:", len(objective))
print("Expected:", total_variables)

print("\nNon-zero objective coefficients:",
      np.count_nonzero(objective))

print("Zero objective coefficients:",
      np.sum(objective == 0))

print("\nExample:")
example_flight = flight_variables.iloc[0]

print(
    "Flight:", example_flight["flight_id"]
)
print(
    "Cancellation coefficient:",
    objective[int(example_flight["cancel_variable_id"])]
)
print(
    "Delay coefficient:",
    objective[int(example_flight["delay_variable_id"])]
)

print("\nOBJECTIVE VECTOR BUILT")

BUILDING OBJECTIVE VECTOR
Objective length: 472798
Expected: 472798

Non-zero objective coefficients: 1578
Zero objective coefficients: 471220

Example:
Flight: FL-1000
Cancellation coefficient: 690440.0
Delay coefficient: 665.0

OBJECTIVE VECTOR BUILT


In [111]:
# PRESOLVER — CELL 67
# Final MILP consistency check

print("=" * 60)
print("FINAL PRESOLVER VALIDATION")
print("=" * 60)

checks = {}

checks["A_rows_match_bounds"] = (
    A.shape[0] == len(constraint_lower) == len(constraint_upper)
)

checks["A_columns_match_variables"] = (
    A.shape[1] == total_variables
)

checks["objective_length_correct"] = (
    len(objective) == total_variables
)

checks["lower_bound_length_correct"] = (
    len(variable_lower) == total_variables
)

checks["upper_bound_length_correct"] = (
    len(variable_upper) == total_variables
)

checks["integrality_length_correct"] = (
    len(variable_integrality) == total_variables
)

checks["variable_bounds_valid"] = (
    np.all(variable_lower <= variable_upper)
)

checks["integrality_valid"] = (
    np.all(np.isin(variable_integrality, [0, 1]))
)

checks["A_has_no_nan"] = (
    not np.isnan(A.data).any()
)

checks["objective_has_no_nan"] = (
    not np.isnan(objective).any()
)

checks["variable_bounds_have_no_nan"] = (
    not np.isnan(variable_lower).any()
    and not np.isnan(variable_upper).any()
)

checks["constraint_bounds_have_no_nan"] = (
    not np.isnan(constraint_lower).any()
    and not np.isnan(constraint_upper).any()
)

for name, result in checks.items():
    print(f"{name}: {result}")

print("\n" + "=" * 60)

if all(checks.values()):
    print("ALL FINAL PRESOLVER CHECKS PASSED")
    print("=" * 60)
    print("\nMILP READY FOR SOLVER HANDOFF")
else:
    print("WARNING: SOME CHECKS FAILED")
    print("=" * 60)

FINAL PRESOLVER VALIDATION
A_rows_match_bounds: True
A_columns_match_variables: True
objective_length_correct: True
lower_bound_length_correct: True
upper_bound_length_correct: True
integrality_length_correct: True
variable_bounds_valid: True
integrality_valid: True
A_has_no_nan: True
objective_has_no_nan: True
variable_bounds_have_no_nan: True
constraint_bounds_have_no_nan: True

ALL FINAL PRESOLVER CHECKS PASSED

MILP READY FOR SOLVER HANDOFF


In [112]:
# PRESOLVER — CELL 68
# Clean handoff package for the solver

solver_input = OptimizationModel(
    A=A,
    constraint_lower=constraint_lower,
    constraint_upper=constraint_upper,
    objective=objective,
    variable_lower=variable_lower,
    variable_upper=variable_upper,
    variable_integrality=variable_integrality
)

print("OptimizationModel created successfully")
print(f"Variables: {solver_input.n_variables}")
print(f"Constraints: {solver_input.n_constraints}")
print(f"Non-zeros: {solver_input.nnz}")
print(f"Problem type: {solver_input.problem_type}")

OptimizationModel created successfully
Variables: 472798
Constraints: 3443145
Non-zeros: 7898666
Problem type: MILP


In [113]:
# PRESOLVER — CELL 69
# Save complete MILP handoff as a single .pkl file

import pickle

solver_handoff = {
    # MILP matrix
    "A": A,

    # Constraint bounds
    "constraint_lower": constraint_lower,
    "constraint_upper": constraint_upper,

    # Objective
    "objective": objective,

    # Variable bounds and types
    "variable_lower": variable_lower,
    "variable_upper": variable_upper,
    "variable_integrality": variable_integrality,

    # Problem dimensions
    "n_variables": total_variables,
    "n_constraints": len(all_constraint_rows),
    "nnz": A.nnz,

    # Original model information
    "flight_variables": flight_variables,
    "aircraft_variables": aircraft_variables,
    "crew_variables": crew_variables,
    "aircraft_arc_variables": aircraft_arc_variables,

    # Constraint metadata
    "constraint_rows": all_constraint_rows
}

with open("solver_handoff.pkl", "wb") as f:
    pickle.dump(solver_handoff, f, protocol=pickle.HIGHEST_PROTOCOL)

print("Solver handoff saved successfully.")
print("File: solver_handoff.pkl")
print(f"Variables:   {n_variables if 'n_variables' in globals() else total_variables:,}")
print(f"Constraints: {len(all_constraint_rows):,}")
print(f"Nonzeros:    {A.nnz:,}")

Solver handoff saved successfully.
File: solver_handoff.pkl
Variables:   472,798
Constraints: 3,443,145
Nonzeros:    7,898,666
